# Phase 19 - Was die Uhr sehen kann

**Braucht eine A100**, ~45 min.

Die Ausgangsidee: statt absoluter Zeitpunkte eine **Phasenkoordinate** φ(t) ∈ [0,2π) innerhalb
des Modellrhythmus, dann P(φ | e) je Experte, schichtkonditioniert als
Δ_e(φ) = P(φ|e,L) − P(φ|L), zuletzt der Vergleich der Arme.

## Was davon hier nicht messbar ist

**Kein Träger.** Eine Phase braucht etwas Periodisches. Beim autoregressiven Dekodieren
wiederholt sich genau ein Ereignis — das Token. Der einzige Träger, den der Mechanismus
hergibt, hat die Periode 1, und bei Periode 1 *ist* „Phase innerhalb der Periode" die Position
im Vorwärtslauf. Ob es darüber hinaus einen gibt, wird in Abschnitt 9 **gemessen**, nicht
angenommen.

**Die feinste trennbare Zeitkoordinate ist die Schicht.** Innerhalb eines Tokens läuft jede
Schicht genau einmal, und alle acht Experten einer Schicht werden am selben Punkt
abgearbeitet. Δ_e(φ|L) hat keine Restvarianz, die dem Experten gehören könnte — was dort
herauskäme, wäre Tokenselektion: „e wurde geroutet" wählt Tokens mit anderem Inhalt und
anderer Kontextlänge aus, also genau den Störfaktor, den die Konditionierung entfernen sollte.

**Der Pfeil zeigt andersherum.** „Experte e bevorzugt die Hochlastphase" braucht
Last → Routing. Routing ist aber eine Funktion von Eingabe und Gewichten. Abschnitt 5 misst
die Kippzahl unter erzwungener Konkurrenz über alle 10 240 Slots.

## Was stattdessen gemessen wird

Der MoE-Vorwärtslauf ist eine **Python-Schleife** über die getroffenen Experten. Beim
Dekodieren sind das genau acht je Schicht — egal welche. Ein Experte ist **6,00 MiB**
(`gate_up_proj[e]` 4 MiB + `down_proj[e]` 2 MiB), ein Schritt adressiert damit rund **2 GB**
Expertengewicht, **unabhängig von der Auswahl**. Zwischen zwei Berührungen desselben Paares
strömen über 3,5 GB durch einen 40-MB-L2 — 88-facher Umschlag. Cache-Wiederverwendung über
benachbarte Tokens ist rechnerisch tot; das war die naheliegende Rettung der Idee, und sie
fällt.

Was bleibt, ist die **Vielfalt innerhalb eines Schrittes**. Im Stapel kann eine Schicht
zwischen 8 und 8·B verschiedene Experten treffen, und jede zusätzliche kostet eine eigene
Schleifeniteration. Das ist **erzwingbar** — über denselben Hakenmechanismus, mit dem seit
Phase 12 maskiert wird — und damit kausal statt beobachtend.

| | Frage | Ergebnis |
|---|---|---|
| **H1** | folgt die Zeit der *Zahl* verschiedener Experten? | κ in ms je Einheit S |
| **H2** | bei gleicher Zahl: bewegen die *Namen* die Uhr? | Schranke in µs, % und ε |
| **H3** | unterscheidet sich der Lastzähler S(t) je Arm? | rauschfrei, aus dem Routing |
| **H4** | gibt es überhaupt einen Träger? | sonst ist φ nicht definiert |
| **H5** | liest der Prefill dasselbe Routing wie das Dekodieren? | prüft Phasen 14–18 |

## Die Eichungen stehen vor den Fragen

- **ε** — eine bekannte Verzögerung wird in eine zufällige Hälfte der Schritte eingespeist. Die
  kleinste, die wiedergefunden wird, ist die Auflösung. Ohne sie ist kein Nullbefund zulässig.
- **Wiederholbarkeit** — Split-half des Zeitprofils **nach Trendabzug**. Ohne Abzug misst die
  Zahl nur den wachsenden Schlüssel-Wert-Speicher und läge auch bei reinem Rauschen nahe eins.
- **Die Attrappe** — `Maske` aus Phase 12 setzt nur *Gewichte* auf null und lässt den
  Befehlsstrom unangetastet: dieselben Gruppen, dieselben Kernel, dieselben 2 GB. Sie ändert
  das Verhalten vollständig (100 → 9 %) und die Arbeit gar nicht. Ihre erwartete Zeitwirkung
  ist genau der Hakenfußabdruck — deshalb ist sie hier eine Kontrolle, keine Behandlung.

Alle Zeitmessungen laufen **lehrergeführt** über einen einmal erzeugten, festen Tokenstrom:
Inhalt, Länge und KV-Wachstum sind zwischen allen Bedingungen bitgleich, auch dann, wenn
erzwungenes Routing die Ausgabe zu Unsinn macht.

## Was der Lauf nicht sagen kann

Nichts über Qwen3.6. Die Zeit je Schritt ist eine Eigenschaft der **Laufzeit**: dieselbe
Auswahl unter einem fusionierten Kernel, unter CUDA-Graphen oder unter vLLM ergäbe einen
anderen Takt. Nichts unterhalb von ε. Nichts über eine Phase unterhalb der Schicht — die gibt
es nicht. Und H1/H2 sagen nichts über Verhalten: erzwungenes Routing zerstört die Ausgabe,
das ist sein Zweck.


In [ ]:
# Welcher Durchgang? 0 ist der erste Lauf, jede andere Zahl ein eigenstaendiger
# Nachlauf mit neuen Ziehungen. Der Tokenstrom wird jedes Mal neu erzeugt.
WIEDERHOLUNG = 0
# === PHASE 19 - WAS DIE UHR SEHEN KANN ======================================
# Die Ausgangsidee: statt absoluter Zeitpunkte eine Phasenkoordinate
# phi(t) in [0,2pi) innerhalb des Modellrhythmus, dann P(phi | e) je Experte,
# schichtkonditioniert als Delta_e(phi) = P(phi|e,L) - P(phi|L), und zuletzt
# der Vergleich der Arme.
#
# WAS DAVON HIER NICHT MESSBAR IST, UND WARUM
#
#   KEIN TRAEGER. Eine Phase braucht etwas Periodisches. Beim autoregressiven
#   Dekodieren wiederholt sich genau ein Ereignis - das Token. Der einzige
#   Traeger, den der Mechanismus hergibt, hat die Periode 1, und bei Periode 1
#   IST "Phase innerhalb der Periode" die Position im Vorwaertslauf. Ob es
#   darueber hinaus einen gibt, wird in Abschnitt 9 gemessen, nicht
#   angenommen.
#
#   DIE FEINSTE TRENNBARE ZEITKOORDINATE IST DIE SCHICHT. Innerhalb eines
#   Tokens laeuft jede Schicht genau einmal, und alle acht Experten einer
#   Schicht werden am selben Punkt abgearbeitet. Delta_e(phi|L) hat deshalb
#   keine Restvarianz, die dem Experten gehoeren koennte. Was dort
#   herauskaeme, waere Tokenselektion - "e wurde geroutet" waehlt Tokens mit
#   anderem Inhalt und anderer Kontextlaenge aus, also genau den Stoerfaktor,
#   den die Konditionierung entfernen sollte. Tiefe gegen Zeit ist ausserdem
#   in Phase 17 schon rauschfrei gemessen.
#
#   DER PFEIL ZEIGT ANDERSHERUM. "Experte e bevorzugt die Hochlastphase"
#   braucht Last -> Routing. Routing ist aber eine Funktion von Eingabe und
#   Gewichten; Taktrate und Temperatur gehen ins top-k nicht ein. Abschnitt 5
#   misst die Kippzahl unter erzwungener Konkurrenz und weist sie aus.
#
# WAS STATTDESSEN GEMESSEN WIRD
#   Der MoE-Vorwaertslauf ist eine PYTHON-SCHLEIFE ueber die getroffenen
#   Experten. Beim Dekodieren sind das genau acht je Schicht - egal welche.
#   Ein Experte ist 6.00 MiB, ein Schritt adressiert damit rund 2 GB
#   Expertengewicht, unabhaengig von der Auswahl. Zwischen zwei Beruehrungen
#   desselben Paares stroemen ueber 3.5 GB durch einen 40-MB-L2; Wiederver-
#   wendung ueber benachbarte Tokens ist rechnerisch tot. Das war die
#   naheliegende Rettung der Idee, und sie faellt.
#
#   Was bleibt, ist die VIELFALT innerhalb eines Schrittes. Im Stapel kann
#   eine Schicht zwischen acht und 8*B verschiedene Experten treffen, und jede
#   zusaetzliche kostet eine eigene Schleifeniteration. Das ist erzwingbar -
#   ueber denselben Hakenmechanismus, mit dem seit Phase 12 maskiert wird -
#   und damit KAUSAL statt beobachtend.
#
#   H1  folgt die Zeit der ZAHL verschiedener Experten?   kappa
#   H2  bei gleicher Zahl: bewegen die NAMEN die Uhr?     Schranke
#   H3  unterscheidet sich der Lastzaehler S(t) je Arm?   rauschfrei
#   H4  gibt es ueberhaupt einen Traeger?                 sonst kein phi
#   H5  liest der Prefill dasselbe Routing wie das Dekodieren?
#
# Die Eichungen stehen VOR den Fragen: EPSILON (was sieht die Uhr ueberhaupt),
# Wiederholbarkeit des Zeitprofils, und ob die Sonde selbst das Signal ist.
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
import re, math, torch, collections, unicodedata, random
import numpy as np, glob, json, gc, sys, time
gc.collect(); torch.cuda.empty_cache()
try: torch.cuda.synchronize()
except Exception: pass
_free=torch.cuda.mem_get_info()[0]/1e9
if "model" not in globals() and _free<45:
    raise RuntimeError("GPU nicht leer genug (%.1f GB frei, ~45 noetig)."%_free)
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
# ---------------- Protokoll und Abbildungen automatisch nach Drive ----------
# Der PDF-Export von Colab schneidet die Ausgabe unzuverlaessig ab. Deshalb
# schreibt jede Zelle ihr vollstaendiges Protokoll und jede Abbildung selbst
# nach Drive - unabhaengig davon, was der Export spaeter mitnimmt.
import sys, time
WC_RUN=globals().get("WC_RUN","phase19_taktgeber")
RUN_OUT="/content/drive/MyDrive/WeirdChat_Runs/%s_%s"%(WC_RUN,time.strftime("%Y%m%d-%H%M%S"))
os.makedirs(RUN_OUT,exist_ok=True)
class _WCTee:
    """Schreibt alles doppelt: in die Zelle und nach Drive. Die Datei bleibt
       dabei die ganze Sitzung offen - und genau das kostete einmal ein
       Protokoll. Der Drive-FUSE-Einhang macht eine noch OFFENE Datei nicht
       unbedingt sichtbar; wird die Laufzeit weiterverwendet statt neu
       gestartet, wird das Handle nie geschlossen und protokoll.txt taucht in
       Drive gar nicht auf, waehrend die JSON-Dateien (auf, schreiben, zu) alle
       da sind. Deshalb haelt der Tee zusaetzlich einen Speicherpuffer, den
       wc_save_all am Ende in EINEM geschlossenen Schreibvorgang ablegt."""
    _wc_tee=True
    def __init__(self,p,o): self.o=o; self.f=None; self.puffer=[]; self.retarget(p)
    def retarget(self,p):
        try:
            if self.f: self.f.close()
        except Exception: pass
        self.pfad=p
        if not hasattr(self,"puffer"): self.puffer=[]
        self.puffer=[]
        try: self.f=open(p,"a",encoding="utf-8")
        except Exception: self.f=None
    def write(self,s):
        self.o.write(s)
        try: self.puffer.append(s)
        except Exception: pass
        if self.f:
            try: self.f.write(s); self.f.flush()
            except Exception: pass
        return len(s)
    def flush(self):
        self.o.flush()
        if self.f:
            try: self.f.flush()
            except Exception: pass
    def isatty(self): return False
_wc_log=os.path.join(RUN_OUT,"protokoll.txt")
if getattr(sys.stdout,"_wc_tee",False): sys.stdout.retarget(_wc_log)
else: sys.stdout=_WCTee(_wc_log,sys.stdout)
try:
    import matplotlib.pyplot as _wcplt
    if not getattr(_wcplt,"_wc_patched",False):
        _wc_orig_show=_wcplt.show; _wc_fig=[0]
        def _wc_show(*a,**k):
            for _num in _wcplt.get_fignums():
                _wc_fig[0]+=1
                try:
                    _wcplt.figure(_num).savefig(os.path.join(RUN_OUT,"abb_%02d.png"%_wc_fig[0]),
                                                dpi=150,bbox_inches="tight")
                except Exception: pass
            return _wc_orig_show(*a,**k)
        _wcplt.show=_wc_show; _wcplt._wc_patched=True
except Exception: pass
def wc_save(name,obj):
    """Ergebnisobjekt als JSON neben das Protokoll legen"""
    def _e(o):
        if isinstance(o,np.ndarray): return o.tolist()
        if isinstance(o,(np.integer,)): return int(o)
        if isinstance(o,(np.floating,)): return float(o)
        if isinstance(o,(np.bool_,)): return bool(o)
        return str(o)
    try:
        with open(os.path.join(RUN_OUT,name+".json"),"w",encoding="utf-8") as f:
            json.dump(obj,f,ensure_ascii=False,indent=1,default=_e)
        print("gespeichert: %s.json"%name)
    except Exception as _ex: print("konnte %s nicht speichern: %s"%(name,_ex))
def wc_protokoll_ablegen():
    """Das Protokoll aus dem Speicherpuffer in EINEM geschlossenen Vorgang
       ablegen. Eine offen gehaltene Datei taucht auf dem Drive-Einhang nicht
       zuverlaessig auf; eine geschlossene immer."""
    try:
        _t=sys.stdout
        if getattr(_t,"_wc_tee",False) and getattr(_t,"puffer",None) is not None:
            _p=os.path.join(RUN_OUT,"protokoll_kopie.txt")
            with open(_p,"w",encoding="utf-8") as _f: _f.write("".join(_t.puffer))
            return _p
    except Exception as _ex:
        print("Protokollkopie fehlgeschlagen: %s"%_ex)
    return None
def wc_save_all():
    """alle *_RESULTS aus dem Namensraum sichern - Aufruf am Zellenende"""
    for _k in [k for k in list(globals()) if k.endswith("_RESULTS")]:
        wc_save(_k,globals()[_k])
    _p=wc_protokoll_ablegen()
    if _p: print("Protokollkopie:",os.path.basename(_p))
    print("Lauf-Ordner:",RUN_OUT)
print("Lauf-Ordner (Protokoll + Abbildungen):",RUN_OUT)
if "PROMPTS" not in globals():
    _h=glob.glob("/content/drive/MyDrive/**/weird_transcripts.jsonl",recursive=True)
    assert _h, "weird_transcripts.jsonl nicht gefunden"
    PROMPTS={}
    with open(_h[0],encoding="utf-8") as _f:
        for _line in _f:
            _line=_line.strip()
            if not _line: continue
            _r=json.loads(_line)
            _pid=str(_r["id"]).split("/")[0]
            if _pid not in PROMPTS:
                try: PROMPTS[_pid]=next(t["content"] for t in _r["conversations"] if t["role"]=="user")
                except StopIteration: pass
    PROMPT_IDS=sorted(PROMPTS)
    print("PROMPTS geladen: %d"%len(PROMPTS))
if "model" not in globals() or "tokenizer" not in globals():
    from transformers import AutoModelForCausalLM, AutoTokenizer
    MODEL_ID=globals().get("MODEL_ID","Qwen/Qwen3.6-35B-A3B-FP8")
    print("lade Instruct-Modell:",MODEL_ID,"(einige Minuten)")
    tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
    model=AutoModelForCausalLM.from_pretrained(MODEL_ID,device_map="auto",torch_dtype="auto")
    model.eval()
    print("geladen | dtype:",next(model.parameters()).dtype)
# ---------------- reine Logik (offline geprueft) ----------------------------
PHRASE="each service's local name"
FRW=[(0x0370,0x03FF),(0x0400,0x052F),(0x0530,0x058F),(0x0590,0x05FF),(0x0600,0x074F),
     (0x0900,0x097F),(0x0E00,0x0E7F),(0x3040,0x30FF),(0x3400,0x9FFF),(0xAC00,0xD7AF),
     (0xF900,0xFAFF)]
JPW=[(0x3040,0x30FF),(0x3400,0x9FFF),(0xF900,0xFAFF)]
FRS=set("le la les une un des est et pour avec dans votre vous voici bonjour du qui que "
        "sur cette ces aux ou par plus il elle nous sont".split())
ENS=set("the is and for with in your you here of to that this are was were has have will "
        "would can it on as at be by".split())
PTES=set("nome nomes servico servicos armazenamento limite limites preco mes gratuito "
         "conta cada para com uma nao mais seu sua nombre servicio servicios "
         "almacenamiento precio cuenta los las del con mas su".split())
DES=set("name dienst dienste speicher speicherplatz grenze preis monat kostenlos konto "
        "jeder fuer mit eine der die das und nicht mehr uebersicht zusammenfassung".split())
def _in(c,bereiche):
    o=ord(c); return any(a<=o<=b for a,b in bereiche)
def _fremd(s):
    return [c for c in s if c.isalpha() and ord(c)>=0x250 and _in(c,FRW)]
def _srun(t,run=3):
    c=0
    for ch in t:
        if ch.isalpha() and ord(ch)>=0x250 and _in(ch,FRW):
            c+=1
            if c>=run: return True
        elif ch.isalpha(): c=0
    return False
def _entakz(s):
    return "".join(c for c in unicodedata.normalize("NFD",s) if not unicodedata.combining(c))
def classify_answer(t):
    if not t.strip(): return "empty"
    al=[c for c in t if c.isalpha()]; fo=_fremd(t)
    if al and len(fo)/len(al)>=0.5: return "takeover"
    if _srun(t): return "gloss"
    w=re.findall(r"[a-zA-ZÀ-ſ']+",t.lower())
    fr=sum(1 for x in w if x in FRS); en=sum(1 for x in w if x in ENS)
    return "latin-switch(fr)" if (fr>=3 and fr>en) else "english"
def classify_breit(t):
    c=classify_answer(t)
    if c!="english": return c
    w=re.findall(r"[a-zA-ZÀ-ſ']+",_entakz(t).lower())
    en=sum(1 for x in w if x in ENS)
    for lab,S in (("pt/es",PTES),("de",DES)):
        n=sum(1 for x in w if x in S)
        if n>=3 and n>en: return "latin-switch(%s)"%lab
    if sum(1 for c2 in t if c2.isalpha() and 0xC0<=ord(c2)<=0x17F)>=3: return "latin-akzent"
    return "english"
SW=("takeover","gloss","latin-switch(fr)")
SWB=SW+("latin-switch(pt/es)","latin-switch(de)","latin-akzent")
def ist_jp(t,mindest=3):
    """Eigener Zaehler fuer die Positivkontrolle. Im JP-Arm ist japanische
       Schrift das ERWUENSCHTE Verhalten - classify_breit wuerde sie
       'takeover' nennen, was hier irrefuehrend waere. Gezaehlt wird ein Lauf
       von mindestens 3 Kana-/Kanji-Zeichen: einzelne Zeichen kommen auch in
       englischen Antworten als Beispiel vor, ein Lauf nicht."""
    c=0
    for ch in t:
        if _in(ch,JPW):
            c+=1
            if c>=mindest: return True
        elif ch.isalpha(): c=0
    return False
def wiederholt(t,fenster=12,mal=4):
    """Zerfallsmerkmal: dieselbe Zeichenfolge viermal. Bei starker Beschaedigung
       faellt ein Modell in Schleifen, lange bevor es verstummt."""
    if len(t)<fenster*mal: return False
    z=collections.Counter(t[i:i+fenster] for i in range(len(t)-fenster+1))
    return max(z.values())>=mal
def zerfall(texte):
    """Was sagt die Antwortform ueber den Schaden - unabhaengig von der Sprache"""
    if not texte: return dict(leer=0.0,laenge=0.0,schleife=0.0)
    return dict(leer=sum(1 for t in texte if not t.strip())/len(texte),
                laenge=sum(len(t) for t in texte)/len(texte),
                schleife=sum(1 for t in texte if wiederholt(t))/len(texte))
def wilson(k,n,z=1.96):
    if n==0: return (0.,0.,0.)
    p=k/n; d=1+z*z/n; c=p+z*z/(2*n); h=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))
    return p,(c-h)/d,(c+h)/d
def fisher2x2(a,b,c,d,einseitig=False):
    from math import lgamma,exp
    lf=lambda n: lgamma(n+1); n=a+b+c+d
    def pr(x):
        y=a+b-x; z=a+c-x; w=n-x-y-z
        if min(y,z,w)<0: return 0.0
        return exp(lf(a+b)+lf(c+d)+lf(a+c)+lf(b+d)-lf(n)-lf(x)-lf(y)-lf(z)-lf(w))
    hi=min(a+b,a+c)
    if einseitig: return min(1.0,sum(pr(x) for x in range(a,hi+1)))
    p0=pr(a)
    return min(1.0,sum(pr(x) for x in range(0,hi+1) if pr(x)<=p0*(1+1e-9)))
def phrase_mit(w):
    return "each service's name" if not w else "each service's %s name"%w
def setze_arm(text,ersatz):
    if PHRASE not in text: return text,False
    return text.replace(PHRASE,ersatz),True
def massstab(*vektoren):
    """EIN globaler, robuster Massstab fuer alle Einheiten. Ersetzt den
       Nenner je Einheit aus v2, der auf 1e-8 fallen konnte und damit
       Trennwerte von 1e7 erzeugt hat. Median statt Mittelwert, weil die
       Zwischenschicht duennbesetzt ist und wenige grosse Werte den Mittelwert
       tragen wuerden. Nullen zaehlen NICHT mit: bei 70% strukturellen Nullen
       waere der Median sonst selbst null."""
    v=np.abs(np.concatenate([np.asarray(x,dtype=np.float64).ravel() for x in vektoren]))
    v=v[v>0]
    if v.size==0: return 1.0
    m=float(np.median(v))
    return m if m>0 else 1.0
def trennung_zwei(xa,xb,s):
    """Differenz zweier Zustaende in Einheiten EINES globalen Massstabs.
       Positiv = im ersten Zustand hoeher. Beschraenkt und vergleichbar."""
    return (np.asarray(xa,dtype=np.float64)-np.asarray(xb,dtype=np.float64))/float(s)
def waehle_einheiten(d,k):
    """die k Einheiten mit der groessten POSITIVEN Trennung - Richtung ist
       vorregistriert: im JP-Zustand hoeher, Ablation muss die JP-Rate senken"""
    return list(np.argsort(-np.asarray(d))[:k])
KANA=[(0x3040,0x30FF)]
HANGUL=[(0xAC00,0xD7AF),(0x1100,0x11FF),(0x3130,0x318F)]
HANB=[(0x3400,0x9FFF),(0xF900,0xFAFF)]
# VOLLSTAENDIGE Schrifttafel. Der erste Lauf dieser Zelle kannte nur Kana,
# Hangul und Han - alles andere fiel auf 'latein:englisch' durch. Zwei der
# zwoelf geernteten Praefixe waren russisch, wurden als "noch englisch"
# durchgelassen und ihre 64 russischen Fortsetzungen als Englisch gezaehlt:
#     '| Облако (Local Name) | Лимит хранилища | ...'  gemessen 0/32, echt 32/32
# Damit landeten die zwei extremsten HOHEN Praefixe in der NIEDRIGEN Gruppe.
# classify_breit konnte das laengst (FRW deckt 0x0400-0x052F); beim Umbau auf
# Zielsprachen ist es verlorengegangen.
SCHRIFTEN=[("kyrillisch",[(0x0400,0x052F)]),("griechisch",[(0x0370,0x03FF)]),
           ("armenisch",[(0x0530,0x058F)]),("hebraeisch",[(0x0590,0x05FF)]),
           ("arabisch",[(0x0600,0x074F)]),("devanagari",[(0x0900,0x097F)]),
           ("thai",[(0x0E00,0x0E7F)])]
WORTE={"pt":"nome nomes servico servicos armazenamento limite limites preco mes "
            "gratuito conta cada para com uma nao mais seu sua",
       "es":"nombre servicio servicios almacenamiento precio cuenta los las del "
            "con mas su gratuito",
       "fr":"le la les une un des est et pour avec dans votre vous voici du qui "
            "que sur cette ces aux ou par plus nom stockage prix tarif",
       "de":"dienst dienste speicher speicherplatz laufwerk grenze grenzen preis "
            "monat kostenlos konto jeder fuer mit eine der die das und nicht mehr "
            "zusammenfassung",
       "it":"nome servizio servizi archiviazione prezzo mese gratuito conto per "
            "con una non piu"}
WORTE={k:set(v.split()) for k,v in WORTE.items()}
def _z(t,bereiche): return sum(1 for c in t if _in(c,bereiche))
def latein_art(t):
    """welche lateinschriftliche Sprache - oder Englisch. 'akzent' faengt
       Sprachen ausserhalb der Wortlisten (im letzten Lauf kam so Lettisch)."""
    w=re.findall(r"[a-zA-Z']+",_entakz(t).lower())
    tr=sorted(((n,sum(1 for x in w if x in S)) for n,S in WORTE.items()),key=lambda x:-x[1])
    if tr[0][1]>=3: return tr[0][0]
    if sum(1 for c in t if c.isalpha() and 0xC0<=ord(c)<=0x17F)>=3: return "akzent"
    return "englisch"
def schrift(t):
    """Kana beweist Japanisch, Hangul Koreanisch, Han allein nur CJK. Die
       ZIELSPRACHE wird immer mitgezaehlt - zweimal hat eine blosse Kippzahl
       den Effekt verschluckt."""
    if not t.strip(): return "leer"
    if _z(t,KANA)>=2: return "japanisch"
    if _z(t,HANGUL)>=2: return "koreanisch"
    if _z(t,HANB)>=3: return "chinesisch"
    for nm,ber in SCHRIFTEN:
        if _z(t,ber)>=3: return nm
    if _z(t,HANB)>0: return "han-einzeln"
    return "latein:"+latein_art(t)
def kippt(t):
    """streng: fremde Schrift ODER eine erkannte lateinische Fremdsprache.
       Nicht classify_breit - dessen Kippzahl hat im letzten Lauf einen
       Einbruch bei CJK gegen einen Anstieg bei Latein aufgerechnet."""
    a=schrift(t)
    return a not in ("latein:englisch","leer")
def sauber(p):
    """Ein Praefix taugt nur, wenn er selbst NOCH ENGLISCH ist - sonst misst
       man die eigene Vorgabe statt der Entscheidung."""
    return bool(p.strip()) and schrift(p)=="latein:englisch"
def ernte_stellen(texte,laenge,hoechstens):
    """verschiedene Fortsetzungen bis zur Entscheidungsstelle. Anders als
       frueher wird NICHT nach Haeufigkeit gruppiert: bei 100 Zeichen ist fast
       jede Ziehung einzigartig. Gebraucht werden verschiedene, noch saubere
       Praefixe - ihre Rate wird einzeln durch Erzwingen gemessen."""
    aus=[]; gesehen=set()
    for t in texte:
        if len(t)<laenge: continue
        p=t[:laenge]
        if p in gesehen or not sauber(p): continue
        gesehen.add(p); aus.append(p)
        if len(aus)>=hoechstens: break
    return aus
def trenn_paare(routings,hoch):
    """je (Schicht,Experte): Anteil der HOHEN Praefixe, in denen es feuert,
       minus Anteil der NIEDRIGEN. +1 heisst 'in allen hohen, in keinem
       niedrigen'. Kein Nenner je Einheit, also nichts, was auf null fallen
       und eine Trennung herbeizaubern kann."""
    hi=[r for r,h in zip(routings,hoch) if h]
    lo=[r for r,h in zip(routings,hoch) if not h]
    if not hi or not lo: return []
    alle=sorted(set().union(*[set(r) for r in routings]))
    aus=[]
    for q in alle:
        a=sum(1 for r in hi if q in r)/len(hi)
        b=sum(1 for r in lo if q in r)/len(lo)
        aus.append((q,a-b))
    aus.sort(key=lambda x:-x[1])
    return aus
def perfekte(paare,schwelle=1.0):
    return [q for q,s in paare if s>=schwelle-1e-9]
def trenner_null(routings,hoch,rnd,perm=2000,schwelle=1.0):
    """Bei acht Praefixen findet man perfekte Trenner auch rein zufaellig.
       Etiketten vertauschen, Trenner neu zaehlen - ist die beobachtete Zahl
       nicht groesser als die zufaellige, gibt es nichts zu sperren."""
    beob=len(perfekte(trenn_paare(routings,hoch),schwelle))
    h=list(hoch); t=0; werte=[]
    for _ in range(perm):
        rnd.shuffle(h)
        n=len(perfekte(trenn_paare(routings,h),schwelle))
        werte.append(n)
        if n>=beob: t+=1
    return beob,(t+1)/(perm+1.0),float(np.mean(werte)) if werte else 0.0
def null_untergrenze(n,k_hoch,schwelle=1.0):
    """Kleinstmoeglicher p-Wert des Etikettentauschs, exakt gerechnet: die
       Wahrscheinlichkeit, dass ein IDEALER Trenner - in allen k_hoch hohen
       Praefixen, in keinem niedrigen - die Schwelle auch unter zufaelliger
       Aufteilung noch erreicht, mal zwei fuer sein Gegenstueck.

       Zwei Laeufe sind an dieser Zahl gescheitert. Erst mit acht Praefixen und
       Schwelle 1.0: perfekte Trenner, p=0.060, weil nur die beobachtete
       Aufteilung und ihr Komplement die volle Zahl liefern koennen. Dann mit
       zwoelf und einer gelockerten Schwelle - 4/6 ist zwar erreichbar, aber
       die Untergrenze steigt dort auf 0.080, der Test kann 0.05 nicht mehr
       unterschreiten. Die brauchbaren Felder:

           K       1.00   0.83   0.75   0.67   0.50
           12     0.002  0.002  0.002  0.080  0.080
           16     0.000  0.000  0.010  0.010  0.132
           20     0.000  0.000  0.001  0.001  0.023

       Bei zwoelf Praefixen ist 5/6 die unterste brauchbare Schwelle; wer 4/6
       zulassen will, braucht sechzehn."""
    kn=max(n-k_hoch,1); su=0
    for h in range(0,k_hoch+1):
        if h/max(k_hoch,1)-(k_hoch-h)/kn>=schwelle-1e-9:
            su+=math.comb(k_hoch,h)*math.comb(kn,min(k_hoch-h,kn))
    return min(1.0,2.0*su/math.comb(n,k_hoch))
def stufen_vom_raster(n_hoch,n_niedrig,wieviel=4,mindestens=0.5):
    """Die Stufen des Trennwertbildes muessen ERREICHBARE Werte sein. Bei sechs
       gegen sechs sind nur Vielfache von 1/6 moeglich; eine Stufe 0.67 liegt
       knapp ueber 4/6=0.6667 und zaehlt dort null, waehrend %.2f sie als
       '0.67' druckt. Genau so sind im zweiten Lauf vier Trenner verschwunden."""
    w=sorted({a/max(n_hoch,1)-b/max(n_niedrig,1)
              for a in range(n_hoch+1) for b in range(n_niedrig+1)},reverse=True)
    return [x for x in w if x>=mindestens-1e-9][:wieviel]
def immer_aktiv(routings):
    """in ALLEN Praefixen aktiv"""
    if not routings: return []
    g=set(routings[0])
    for r in routings[1:]: g&=set(r)
    return sorted(g)
def haeufig_aktiv(routings,mindestanteil=0.5):
    """Quelle der Zufallskontrolle. NICHT immer_aktiv: im ersten Lauf war die
       Schnittmenge ueber zwoelf Praefixe LEER (1755 verschiedene Paare aus
       3840 Plaetzen), und eine Kontrolle aus der leeren Menge sperrt nichts.
       Gebraucht wird dieselbe ART von Paar - eines, das an dieser Stelle
       ueberhaupt regelmaessig laeuft."""
    if not routings: return []
    z=collections.Counter()
    for r in routings: z.update(set(r))
    n=len(routings)
    return sorted(q for q,k in z.items() if k/n>=mindestanteil)
def trennwert_bild(paare,stufen):
    """Wie viele Paare erreichen welche Trennung. Ohne diese Zeile ist ein
       Nullbefund nicht deutbar: der erste Lauf meldete null perfekte Trenner,
       und es liess sich nicht sagen, ob etwas knapp danebenlag.

       Die Stufen liegen auf dem RASTER. Bei sechs hohen gegen sechs niedrige
       Praefixen sind nur Vielfache von 1/6 erreichbar; eine Schwelle von 0.67
       liegt knapp ueber 4/6 = 0.6667 und schliesst genau die Faelle aus, die
       sie treffen soll. Der zweite Lauf ist daran haengengeblieben: vier Paare
       wurden als '0.67' gedruckt und die Stufe 0.67 zaehlte null."""
    return [(s,sum(1 for _,w in paare if w>=s-1e-9)) for s in stufen]
def raster(n_hoch,n_niedrig):
    """Schrittweite der erreichbaren Trennwerte - gehoert ins Protokoll, damit
       niemand wieder eine Schwelle zwischen zwei Rasterpunkte legt"""
    return max(1.0/max(n_hoch,1),1.0/max(n_niedrig,1))
def nach_schicht(paare):
    d=collections.defaultdict(set)
    for l,e in paare: d[int(l)].add(int(e))
    return dict(d)
def exklusiv(za,zb): return sorted(set(za)-set(zb))
def urteil_dosis(k_bas,n_bas,k_abl,n_abl,alpha=0.05):
    if n_bas==0 or n_abl==0: return "still"
    p=fisher2x2(k_abl,n_abl-k_abl,k_bas,n_bas-k_bas)
    if p>=alpha: return "still"
    return "senkt" if k_abl/n_abl<k_bas/n_bas else "hebt"
def urteil_arm(k_bas,n_bas,k_aus,n_aus,k_zuf,n_zuf,alpha=0.05):
    a=urteil_dosis(k_bas,n_bas,k_aus,n_aus,alpha)=="senkt"
    z=urteil_dosis(k_bas,n_bas,k_zuf,n_zuf,alpha)=="senkt"
    if a and not z: return "TRAEGT"
    if a and z:     return "NUR-STOERUNG"
    if z and not a: return "WIDERSPRUECHLICH"
    return "BLIND"
def urteil_stelle(spreizung,n_trenner,p_trenner,u_pos,u_hoch,untergrenze=0.0,
                  mindest_spreizung=0.3,mindest_trenner=3,alpha=0.05):
    """Reihenfolge ist Absicht. Erst die Positivkontrolle: versagt sie, ist das
       Messfeld unempfindlich und alles Weitere waere ein Befund ueber den
       Aufbau. Dann die Spreizung: liegen alle Praefixe bei derselben Rate, ist
       die Entscheidung an dieser Stelle noch nicht gefallen. Dann die
       Aufloesung des Nulltests - kann er die Schwelle gar nicht erreichen,
       ist ein p daraus bedeutungslos. Dann der Nulltest selbst. Erst danach
       die eigentliche Frage."""
    if u_pos!="senkt": return "MESSFELD-UNEMPFINDLICH"
    if spreizung<mindest_spreizung: return "ZU-WENIG-SPREIZUNG"
    if n_trenner<mindest_trenner: return "ZU-WENIG-TRENNER"
    if untergrenze>=alpha: return "AUFLOESUNG-ZU-GROB"
    if p_trenner>=alpha: return "TRENNER-ZUFAELLIG"
    return {"TRAEGT":"ENTSCHEIDUNGSSTELLE-TRAEGT","NUR-STOERUNG":"NUR-STOERUNG",
            "WIDERSPRUECHLICH":"WIDERSPRUECHLICH","BLIND":"BLIND"}[u_hoch]
BRAILLE=[(0x2800,0x28FF)]
KYR=[(0x0400,0x052F)]
# Hepburn-Umschrift der Dienste aus dem Prompt. Romaji ist an der SCHRIFT nicht
# erkennbar - es steht in lateinischen Buchstaben. Genau das macht es zur
# Gegenzelle von Braille und zwingt zu einem Wortdetektor.
ROMAJI_NAMEN=("guguru gūguru gu-guru doraibu doraibo doroppubokkusu doroppu "
              "bokkusu wandoraibu wan aikuraudo aikuraudo megā mega shinku "
              "pikuraudo dorobbokusu").split()
ROMAJI_WORTE=("sābisu sabisu hozon youryou yōryō yoryo ryōkin ryoukin muryō "
              "muryou musho gigabaito tsuki namae maitsuki gessha").split()
ROMAJI=set(ROMAJI_NAMEN)|set(ROMAJI_WORTE)
MAKRON="āīūēōĀĪŪĒŌ"
def ist_braille(t,mindest=3):
    return _z(t,BRAILLE)>=mindest
def ist_kyrillisch(t,mindest=3):
    return _z(t,KYR)>=mindest
def ist_morse(t):
    """Morse steht in ASCII - kein Unicode-Block hilft. Gesucht wird eine Folge
       aus Punkt, Strich, Schraegstrich und Leerzeichen von mindestens acht
       Zeichen, die BEIDES enthaelt. Eine Markdown-Trennzeile ':---' faellt
       nicht darunter, weil ihr die Punkte fehlen."""
    for m in re.finditer(r"[.\-/ ]{8,}",t):
        s=m.group(0)
        if s.count("-")>=3 and s.count(".")>=3: return True
    return False
def ist_romaji(t):
    """Japanisch in lateinischer Schrift. Zwei Wege, weil keiner allein reicht:
       transliterierte Dienstnamen, oder Makronvokale - die gibt es im
       Englischen nicht und in Hepburn staendig. Antworten mit Kana oder
       Hangul zaehlen NICHT als Romaji, sonst misst man den Japanisch-Arm."""
    if _z(t,KANA)>=2 or _z(t,HANGUL)>=2: return False
    w=set(re.findall(r"[a-zāīūēōâîûêô']+",t.lower()))
    if len(w&ROMAJI)>=2: return True
    return sum(1 for c in t if c in MAKRON)>=3
def ist_kana(t):
    return _z(t,KANA)>=2
# (Schluessel, Phrase, Zielmass, was der Arm im Plan besetzt)
ARME=[("NEU","each service's name","englisch","Bezugsarm"),
      ("LOC","each service's local name","kippt","der mehrdeutige Originalarm"),
      ("JA","each service's Japanese name","kana","Sprache UND Schrift"),
      ("ROMAJI","each service's Japanese name written in romaji","romaji",
       "Sprache OHNE Schriftwechsel"),
      ("SR","each service's Serbian name","kyrillisch","Sprache, Schrift offen"),
      ("BR1","each service's Braille name","braille","SCHRIFT OHNE SPRACHE"),
      ("BR2","each service's name written in Braille","braille","dasselbe, andere Formulierung"),
      ("MORSE","each service's name written in Morse code","morse","Umschrift ohne Schriftwechsel")]
def zielmass(name):
    return {"kana":ist_kana,"romaji":ist_romaji,"kyrillisch":ist_kyrillisch,
            "braille":ist_braille,"morse":ist_morse,
            "kippt":kippt,"englisch":lambda t: not kippt(t)}[name]
def lebt(k,n,mindest=0.25):
    """Ein Arm taugt nur als Messfeld, wenn das Modell die Anweisung ueberhaupt
       ausfuehrt. Der erste Minimalpaar-Lauf ist an einem toten Feld gescheitert
       (5.5% statt 84.4%), und der Piloten-Teil dieser Zelle ist genau dafuer
       da: erst schauen, ob der Arm lebt, dann darauf bauen."""
    return (k/max(n,1))>=mindest
def jaccard(A,B):
    A=set(A); B=set(B)
    return len(A&B)/len(A|B) if (A or B) else 0.0
def exklusiv(za,zb): return sorted(set(za)-set(zb))
def nach_schicht(paare):
    d=collections.defaultdict(set)
    for l,e in paare: d[int(l)].add(int(e))
    return dict(d)
def ueberlappungs_null(A,B,ref,n_experten,rnd,perm=2000):
    """Wie gross waere die Ueberlappung zweier exklusiver Mengen zufaellig? Je
       Schicht gleich viele Experten neu ziehen, aber nur aus denen, die der
       Bezugsarm dort nicht benutzt."""
    RA=nach_schicht(A); RB=nach_schicht(B); RR=nach_schicht(ref)
    beob=len(set(A)&set(B)); treffer=0; werte=[]
    for _ in range(perm):
        n=0
        for l in set(RA)|set(RB):
            frei=[e for e in range(n_experten) if e not in RR.get(l,set())]
            a=rnd.sample(frei,min(len(RA.get(l,())),len(frei)))
            b=rnd.sample(frei,min(len(RB.get(l,())),len(frei)))
            n+=len(set(a)&set(b))
        werte.append(n)
        if n>=beob: treffer+=1
    return beob,(treffer+1)/(perm+1.0),float(np.mean(werte))
def urteil_dosis(k_bas,n_bas,k_abl,n_abl,alpha=0.05):
    if n_bas==0 or n_abl==0: return "still"
    p=fisher2x2(k_abl,n_abl-k_abl,k_bas,n_bas-k_bas)
    if p>=alpha: return "still"
    return "senkt" if k_abl/n_abl<k_bas/n_bas else "hebt"
def urteil_schrift(lebt_ja,lebt_romaji,u_ja,u_romaji):
    """Die Frage, um die es geht: kodiert die JA-exklusive Expertenmenge die
       SPRACHE oder die SCHRIFT? Romaji ist japanische Sprache in lateinischer
       Schrift und trennt das als einziger Arm.

         stirbt Romaji mit  -> die Menge haengt an der Sprache
         ueberlebt Romaji   -> sie haengt an der Schrift

       Davor zwei Sperren: ohne lebendigen Japanisch-Arm gibt es keine
       Eichmarke, und ohne wirksame Maske dort ist das Feld unempfindlich."""
    if not lebt_ja: return "EICHMARKE-FEHLT"
    if u_ja!="senkt": return "MESSFELD-UNEMPFINDLICH"
    if not lebt_romaji: return "ROMAJI-TOT"
    return "MENGE-KODIERT-SPRACHE" if u_romaji=="senkt" else "MENGE-KODIERT-SCHRIFT"
def dosisgleich(ziel_plaetze,zaehler,verboten,rnd,toleranz=0.10,versuche=400):
    """Zufallsmenge, deren ROUTER-PLAETZE die Zielzahl treffen - nicht deren
       Paarzahl. Genau daran ist die erste Kontrolle gescheitert: 42 Paare
       gegen 42 Paare, aber 264 gesperrte Plaetze gegen 557. Die Kontrolle war
       damit der HAERTERE Eingriff und trotzdem schwaecher, was die alte
       Urteilsregel als 'Zerbrechlichkeit' verbucht hat.

       Gierig aufgefuellt, viele Anlaeufe, der beste Treffer gewinnt. Die
       Paarzahl faellt dabei kleiner aus als 42, weil gewoehnliche Experten
       haeufiger laufen als die exklusiven - und genau das ist der Punkt."""
    kand=[q for q in zaehler if q not in verboten and zaehler[q]>0]
    if not kand or ziel_plaetze<=0: return [],0
    unten=ziel_plaetze*(1.0-toleranz); oben=ziel_plaetze*(1.0+toleranz)
    best=None
    for _ in range(versuche):
        rnd.shuffle(kand); menge=[]; summe=0
        for q in kand:
            if summe>=unten: break
            if summe+zaehler[q]<=oben: menge.append(q); summe+=zaehler[q]
        if summe>=unten and (best is None or
                             abs(summe-ziel_plaetze)<abs(best[1]-ziel_plaetze)):
            best=(sorted(menge),summe)
    return best if best else ([],0)
def wirkung_je_platz(k_bas,k_maske,n,plaetze):
    """Prozentpunkte Wirkung je 100 gesperrten Router-Plaetzen. Ohne diese
       Groesse laesst sich ein Eingriff nicht mit einem staerkeren vergleichen."""
    if plaetze<=0 or n<=0: return 0.0
    return 100.0*(100.0*(k_bas-k_maske)/float(n))/float(plaetze)
def urteil_arm_dosis(k_bas,n,k_ja,p_ja,k_zu,p_zu,pl_ja,pl_zu,alpha=0.05,faktor=2.0):
    """NUR-STOERUNG erst, wenn die Kontrolle JE PLATZ aehnlich stark wirkt.
       Die alte Regel fragte nur 'beide signifikant?' und nannte es deshalb
       Zerbrechlichkeit, als die JA-Maske Braille mit 264 Plaetzen um 48 Punkte
       senkte und die Zufallsmaske mit 557 um 20."""
    a=(k_ja<k_bas) and p_ja<alpha
    z=(k_zu<k_bas) and p_zu<alpha
    if not a: return "WIDERSPRUECHLICH" if z else "STILL"
    if not z: return "TRAEGT"
    ej=wirkung_je_platz(k_bas,k_ja,n,pl_ja); ez=wirkung_je_platz(k_bas,k_zu,n,pl_zu)
    return "TRAEGT-UEBERWIEGEND" if ej>=faktor*max(ez,1e-9) else "NUR-STOERUNG"
TRAEGT_ALLE=("TRAEGT","TRAEGT-UEBERWIEGEND")
# ---------------- Was die Uhr sehen kann -----------------------------------
# Die Ausgangsidee war eine Phasenkoordinate phi(t) in [0,2pi) je Feuerereignis
# und daraus P(phi | e). Diese Koordinate gibt es in dieser Architektur nicht,
# und zwar aus drei Gruenden, von denen jeder einzeln genuegt.
#
#   KEIN TRAEGER. Eine Phase braucht etwas Periodisches. Beim autoregressiven
#   Dekodieren wiederholt sich genau ein Ereignis - das Token. Der einzige
#   Traeger, den der Mechanismus hergibt, hat also die Periode 1 Token, und
#   bei Periode 1 IST "Phase innerhalb der Periode" die Position im
#   Vorwaertslauf. Ob es darueber hinaus einen Traeger gibt, wird in H4
#   gemessen und nicht angenommen.
#
#   DIE FEINSTE TRENNBARE ZEITKOORDINATE IST DIE SCHICHT. Innerhalb eines
#   Tokens laeuft jede Schicht genau einmal; alle acht Experten einer Schicht
#   werden am selben Punkt des Durchlaufs abgearbeitet. Delta_e(phi|L) hat
#   deshalb keine Restvarianz, die dem Experten gehoeren koennte - was dort
#   herauskommt, ist Tokenselektion: "e wurde geroutet" waehlt Tokens mit
#   anderem Inhalt und anderer Kontextlaenge aus, also genau den Stoerfaktor,
#   den die Konditionierung entfernen sollte. Tiefe gegen Zeit ist ausserdem
#   in Phase 17 schon rauschfrei gemessen.
#
#   DER PFEIL ZEIGT ANDERSHERUM. Routing ist eine Funktion von Eingabe und
#   Gewichten. Taktrate, Temperatur und Host-Jitter gehen in das top-k des
#   Routers nicht ein. "Experte e bevorzugt die Hochlastphase" braeuchte
#   Last -> Routing, und das kann es nur ueber numerische Nichtdeterminiertheit
#   nahe an Gleichstaenden geben. G0b misst diese Kippzahl und weist sie aus.
#
# WAS STATTDESSEN MESSBAR IST
#   Ein Experte ist 6.00 MiB: gate_up_proj[e] ist (2*512, 2048) bf16 = 4 MiB,
#   down_proj[e] ist (2048, 512) bf16 = 2 MiB. Ein Dekodierschritt bei B=1
#   adressiert 40 * 8 * 6 MiB = 2.01 GB Expertengewicht - UNABHAENGIG davon,
#   welche acht. Zwischen zwei Beruehrungen desselben Paares stroemen ueber
#   3.5 GB durch einen 40-MB-L2, also rund 88-facher Umschlag. Wiederverwendung
#   ueber benachbarte Tokens ist damit rechnerisch tot; das war die erste
#   Rettungsidee und sie faellt. Was bleibt, ist die VIELFALT innerhalb eines
#   Schrittes: bei B>1 kann eine Schicht zwischen 8 und 8*B verschiedene
#   Experten treffen, und jede zusaetzliche kostet eine eigene Gruppe.
def lastzaehler(je_schicht):
    """S(t) = Summe ueber Schichten der Zahl VERSCHIEDENER Experten.

       Rauschfrei: kommt aus dem Routing, nicht aus der Uhr. Bei B=1 ist S
       konstant 320 - deshalb braucht H1 einen Stapel."""
    return sum(len(set(v)) for v in je_schicht)
def entfernen_trend(werte,laengen,grad=1):
    """Die Zeit je Schritt WAECHST mit dem Zusammenhang. Ohne Abzug hiesse
       'hohe Last' schlicht 'spaet in der Antwort', und jede Aussage ueber
       Experten waere die Tiefenkarte aus Phase 12 in neuen Worten."""
    y=np.asarray(werte,dtype=float)
    if len(y)<grad+2: return [0.0]*len(y)
    X=np.vander(np.asarray(laengen,dtype=float),grad+1)
    try: b=np.linalg.lstsq(X,y,rcond=None)[0]
    except Exception: return [float(v) for v in (y-y.mean())]
    return [float(v) for v in (y-X.dot(b))]
def leiter(zeiten,marken,stufen,alpha=0.01,perm=400,rnd=None):
    """Aufloesung der Messkette. In eine zufaellige Haelfte der Schritte wird
       eine bekannte Verzoegerung eingespeist; die kleinste Stufe, die noch
       mit p<alpha wiedergefunden wird, ist EPSILON.

       Ohne diese Zahl ist kein Nullbefund zulaessig. 'Kein Effekt gefunden'
       heisst sonst nur 'unter dem, was die Uhr sieht' - und wie viel das ist,
       weiss man nicht."""
    aus=[]
    for s in stufen:
        a=[z for z,m in zip(zeiten,marken) if m==s]
        b=[z for z,m in zip(zeiten,marken) if m==0]
        if len(a)<3 or len(b)<3: aus.append((s,None,None)); continue
        beob=float(np.median(a)-np.median(b))
        alle=list(a)+list(b); n=len(a)
        r=rnd or random.Random(12345); tref=0
        for _ in range(perm):
            r.shuffle(alle)
            if abs(float(np.median(alle[:n])-np.median(alle[n:])))>=abs(beob): tref+=1
        aus.append((s,beob,(tref+1.0)/(perm+1.0)))
    gut=[s for s,d,p in aus if p is not None and p<alpha and d is not None and d>0]
    return (min(gut) if gut else None),aus
def icc_haelften(laeufe,perm=200,rnd=None):
    """Split-half-Zuverlaessigkeit des ZEITPROFILS ueber Wiederholungen.

       Eine Reihe ohne Wiederholbarkeit kann keinen Armunterschied tragen, bei
       keinem p. Die Haelften werden gemittelt und korreliert; die Null
       vertauscht die Schrittpositionen innerhalb jedes Laufs und zerstoert
       damit genau das Profil, nicht die Verteilung."""
    M=np.asarray(laeufe,dtype=float)
    if M.ndim!=2 or M.shape[0]<4 or M.shape[1]<4: return None,None
    # ERST den Trend heraus, dann korrelieren. Der Anstieg durch den
    # wachsenden Schluessel-Wert-Speicher ist in ALLEN Wiederholungen
    # derselbe; ohne Abzug misst die Zuverlaessigkeit nur ihn und kaeme auch
    # bei reinem Rauschen nahe an eins. Der erste Bau tat das, und die
    # Probewelt ohne jedes Profil erreichte damit ICC 0.99.
    L=np.arange(M.shape[1],dtype=float)
    M=np.vstack([np.asarray(entfernen_trend(row,L),dtype=float) for row in M])
    def r_von(X):
        h=X.shape[0]//2
        a=X[:h].mean(axis=0); b=X[h:2*h].mean(axis=0)
        if a.std()<1e-12 or b.std()<1e-12: return 0.0
        return float(np.corrcoef(a,b)[0,1])
    beob=r_von(M)
    r=rnd or random.Random(7); tref=0
    for _ in range(perm):
        Z=np.vstack([row[r.sample(range(M.shape[1]),M.shape[1])] for row in M])
        if r_von(Z)>=beob: tref+=1
    return beob,(tref+1.0)/(perm+1.0)
def tost(a,b,delta):
    """Gleichwertigkeit statt 'p>0.05, also nichts'.

       Zwei einseitige Tests gegen die vorab festgelegte Schranke delta.
       GLEICHWERTIG heisst: der Unterschied liegt nachweislich innerhalb von
       delta. UNENTSCHIEDEN heisst: die Messung reicht fuer keine der beiden
       Aussagen - und das ist etwas anderes als Gleichheit.

       VERSCHIEDEN verlangt BEIDES: der Punktwert liegt ausserhalb von delta
       UND der Unterschied ist ueberhaupt von null zu trennen. Der erste Bau
       verlangte nur das erste und nannte fuenf gegen fuenf Ziehungen
       'verschieden', sobald der Mittelwertunterschied zufaellig gross genug
       ausfiel."""
    a=np.asarray(a,dtype=float); b=np.asarray(b,dtype=float)
    if len(a)<3 or len(b)<3: return "UNENTSCHIEDEN",None,None
    d=float(a.mean()-b.mean())
    s=math.sqrt(a.var(ddof=1)/len(a)+b.var(ddof=1)/len(b))
    if s<=0: return ("GLEICHWERTIG" if abs(d)<delta else "VERSCHIEDEN"),d,0.0
    def P(z): return 0.5*(1.0+math.erf(z/math.sqrt(2.0)))
    p_gleich=max(1.0-P((d+delta)/s),P((d-delta)/s))
    p_null=2.0*(1.0-P(abs(d)/s))
    if p_gleich<0.05: return "GLEICHWERTIG",d,p_gleich
    if abs(d)>delta and p_null<0.05: return "VERSCHIEDEN",d,p_null
    return "UNENTSCHIEDEN",d,p_gleich
def autokorr1(d):
    """Korrelation zwischen Nachbarschritten - daraus wird die Blocklaenge
       gewaehlt. Wird ausgewiesen, damit die Wahl nicht im Verborgenen
       stattfindet."""
    d=np.asarray(d,dtype=float)
    if len(d)<8: return 0.0
    z=d-d.mean(); s=float((z*z).sum())
    if s<=0: return 0.0
    return float((z[:-1]*z[1:]).sum())/s
def blockmediane(d,laenge):
    """Eine Schrittreihe auf BLOCKmediane verdichten.

       Nachbarschritte haengen zusammen: Allokator, Taktrate, Inhalt. Wer die
       Vorzeichen einzelner Schritte umkehrt, unterstellt Unabhaengigkeit, die
       es nicht gibt."""
    d=list(d); n=max(1,int(laenge))
    return [float(np.median(d[i:i+n])) for i in range(0,len(d),n) if len(d[i:i+n])>=max(2,n//2)]
def gepaart(d,perm=2000,rnd=None):
    """Median der paarweisen Differenzen, Null durch Vorzeichenumkehr.

       ACHTUNG, hier steckte der teuerste Fehler dieses Bauteils. Uebergeben
       werden muss EINE Zahl je UNABHAENGIGER Einheit - je A/B-Laufpaar oder
       je Block -, nicht je Schritt.

       Der erste Bau kehrte die Vorzeichen je Schritt um. In der Probewelt mit
       AR(1)-Jitter (Korrelation 0.7 zwischen Nachbarschritten) ergab das eine
       Fehlalarmrate von 35 % bei nominal 5 %: die Umkehr unterstellt
       austauschbare, unabhaengige Vorzeichen, und autokorrelierte Differenzen
       sind das nicht. Der Median schwankt dann viel staerker, als die Null
       glaubt. Gepaart wird weiterhin nach Schrittposition - das haelt
       Kontextlaenge, Inhalt und Drift zwischen den Bedingungen gleich -, aber
       gezaehlt wird auf der Einheit, die wirklich unabhaengig ist."""
    d=np.asarray(d,dtype=float)
    if len(d)<4: return None,None
    beob=float(np.median(d)); r=rnd or random.Random(99); tref=0
    for _ in range(perm):
        z=d*np.array([1.0 if r.random()<0.5 else -1.0 for _ in range(len(d))])
        if abs(float(np.median(z)))>=abs(beob): tref+=1
    return beob,(tref+1.0)/(perm+1.0)
def blocklaenge(d,hoechstens=24,schranke=0.10):
    """Blocklaenge fuer die BESCHREIBUNG - nicht fuer den Test.

       Der Weg ueber Bloecke sieht verlockend aus und traegt nicht. Gemessen
       in der Probewelt (AR(1)=0.7, schwere Enden, 300 Wiederholungen,
       nominal 5 % Fehlalarm):

         Vorzeichenumkehr je SCHRITT                    33 %
         feste Blocklaenge 8, vorab gesetzt              5 %
         Blocklaenge aus DENSELBEN Daten gewaehlt       11 %

       Die dritte Zeile ist der Punkt. Wer die Blocklaenge an denselben Daten
       waehlt, an denen er testet, sucht sich die kuerzeste Laenge, die
       zufaellig unkorreliert aussieht - und unterschaetzt damit die Streuung
       systematisch. Die Auswahl ist Teil des Verfahrens und gehoert in die
       Null, sonst eicht sie nicht.

       Deshalb testet dieser Lauf auf UNABHAENGIGEN LAUFPAAREN: je Paar eine
       Zahl, Vorzeichenumkehr ueber die Paare. Diese Funktion beschreibt nur,
       wie weit die Abhaengigkeit reicht, und ihr Wert wird ausgewiesen."""
    d=list(d)
    for n in range(2,hoechstens+1):
        b=blockmediane(d,n)
        if len(b)<6: return max(2,n-1)
        if abs(autokorr1(b))<schranke: return n
    return hoechstens
def steigung(x,y,proben=500,rnd=None):
    """kappa: Mikrosekunden je Einheit Vielfalt, mit Bootstrap-Vertrauen.

       Zwei Ausgaenge sind vorab beide informativ. Zahlt die laufende
       Umsetzung je NICHTLEERER GRUPPE, ist kappa positiv und liegt in der
       Groessenordnung 6 MiB * 40 / Bandbreite. Holt sie stattdessen fuer
       jeden Platz eine Zeile, ist kappa null - und das ist keine
       Fehlmessung, sondern die Diagnose 'die Laufzeit zahlt fuer Plaetze,
       nicht fuer verschiedene Experten'."""
    x=np.asarray(x,dtype=float); y=np.asarray(y,dtype=float)
    if len(x)<4 or x.std()<1e-12: return None,None,None
    def s_von(i):
        return float(np.polyfit(x[i],y[i],1)[0])
    beob=s_von(np.arange(len(x)))
    r=rnd or random.Random(5); bs=[]
    for _ in range(proben):
        i=np.array([r.randrange(len(x)) for _ in range(len(x))])
        if x[i].std()<1e-12: continue
        bs.append(s_von(i))
    if not bs: return beob,None,None
    bs.sort()
    return beob,bs[int(0.025*len(bs))],bs[min(len(bs)-1,int(0.975*len(bs)))]
def blocknull(werte,label,perm=2000,rnd=None):
    """Null fuer H1: die Stufenetiketten werden ueber die BLOECKE vertauscht.

       Zerstoert die Zuordnung Vielfalt->Zeit. Behaelt Drift, Autokorrelation
       innerhalb eines Blocks und jeden Host-Effekt, weil ganze Bloecke
       zusammen wandern."""
    x=np.asarray(label,dtype=float); y=np.asarray(werte,dtype=float)
    if len(x)<4 or x.std()<1e-12: return None,None
    beob=float(np.polyfit(x,y,1)[0]); r=rnd or random.Random(11); tref=0
    idx=list(range(len(x)))
    for _ in range(perm):
        r.shuffle(idx)
        if abs(float(np.polyfit(x[idx],y,1)[0]))>=abs(beob): tref+=1
    return beob,(tref+1.0)/(perm+1.0)
def armnull(je_lauf,arme,perm=2000,rnd=None):
    """Null fuer H3: das ARMETIKETT wird ueber die LAEUFE vertauscht.

       Einheit ist der Lauf, nicht das Token. Die 20 Laeufe sind die einzigen
       unabhaengigen Einheiten; 20*96 Tokens waeren geborgte Freiheitsgrade.
       Vertauscht wird das Etikett, erhalten bleiben die Bahn jedes Laufs,
       seine Laenge, sein Inhalt und seine Lage in der Sitzung."""
    a=sorted(set(arme))
    if len(a)<2 or len(je_lauf)<4: return None,None,None
    def spann(lab):
        m=[np.mean([v for v,x in zip(je_lauf,lab) if x==k]) for k in a]
        return float(max(m)-min(m))
    beob=spann(arme); r=rnd or random.Random(13); lab=list(arme); tref=0
    for _ in range(perm):
        r.shuffle(lab)
        if spann(lab)>=beob: tref+=1
    return beob,(tref+1.0)/(perm+1.0),1.0/(perm+1.0)
def traeger(serie,laengen,perm=200,maxlag=12,rnd=None):
    """Gibt es einen Traeger mit Periode > 1 Token?

       Erst Trend entfernen, dann Autokorrelation ab Lag 2. Lag 1 bleibt
       aussen vor: benachbarte Schritte haengen ueber Inhalt und Allokator
       ohnehin zusammen, und ein Gipfel dort waere kein Takt.

       Die Null vertauscht die Residuen INNERHALB des Laufs. Das zerstoert
       jede zeitliche Ordnung und behaelt die Verteilung - genau richtig fuer
       die Frage nach Periodizitaet."""
    res=np.asarray(entfernen_trend(serie,laengen),dtype=float)
    n=len(res)
    if n<3*maxlag: return None,None,None
    def ak(z):
        z=z-z.mean(); s=float((z*z).sum())
        if s<=0: return np.zeros(maxlag)
        return np.array([float((z[:-k]*z[k:]).sum())/s for k in range(1,maxlag+1)])
    A=ak(res); gip=int(np.argmax(A[1:]))+2; wert=float(A[gip-1])
    r=rnd or random.Random(17); nul=[]
    for _ in range(perm):
        z=res[r.sample(range(n),n)]
        nul.append(float(np.max(ak(z)[1:])))
    nul.sort(); sw=nul[min(len(nul)-1,int(0.99*len(nul)))]
    return (gip if wert>sw else None),wert,sw
def kippzahl(a,b):
    """Wie viele (Schicht,Experte)-Plaetze unterscheiden zwei Routings?

       Null erwartete Kippungen heisst: Last beruehrt das Routing nicht, und
       damit ist die urspruengliche Leserichtung erledigt. Beobachtete null
       ist dabei eine SCHRANKE von 1/(10240*Wiederholungen), kein Beweis."""
    return len(set(a)^set(b))
def urteil_takt(eps,icc_p,sonde,kappa_p,ident,traeger_p,gates):
    """Reihenfolge ist Absicht, und die Eichungen stehen ganz vorn.

       Sieht die Uhr eine eingespeiste Verzoegerung nicht, ist keine Aussage
       ueber Mikrosekunden zulaessig. Wiederholt sich das Zeitprofil nicht,
       kann es keinen Armunterschied tragen. Und bewegt die Sonde selbst die
       Uhr, ist jeder gefundene Effekt der eigene Fussabdruck."""
    for name,ok in gates:
        if not ok: return name
    if eps is None: return "UHR-BLIND"
    if sonde=="VERSCHIEDEN": return "SONDE-IST-SIGNAL"
    # Fehlende Wiederholbarkeit sperrt H4 und jede Aussage ueber das PROFIL -
    # aber nicht H1 und H2. Der erste Bau liess sie alles sperren, und in der
    # Probewelt ohne Dosis blieb dadurch nur KEIN-TAKT uebrig: die Antwort
    # "die Zeit sieht nichts" war gar nicht erreichbar. H1 und H2 vergleichen
    # Blockmediane zwischen Bedingungen; dafuer braucht es eine aufloesende
    # Uhr, kein wiederkehrendes Profil.
    ohne_profil=(icc_p is None or icc_p>=0.05)
    sieht_vielfalt=(kappa_p is not None and kappa_p<0.05)
    sieht_ident=(ident=="VERSCHIEDEN")
    if sieht_vielfalt and not sieht_ident: code="ZEIT-SIEHT-VIELFALT-NICHT-IDENTITAET"
    elif sieht_ident: code="ZEIT-SIEHT-IDENTITAET"
    else: code="ZEIT-SIEHT-NICHTS"
    return code+("-OHNE-PROFIL" if ohne_profil else "")
# ---------------- Ausfuehrung ------------------------------------------------
if "PROMPTS" not in globals():
    _h=glob.glob("/content/drive/MyDrive/**/weird_transcripts.jsonl",recursive=True)
    assert _h,"weird_transcripts.jsonl nicht gefunden"
    PROMPTS={}
    with open(_h[0],encoding="utf-8") as _f:
        for _l in _f:
            _l=_l.strip()
            if not _l: continue
            _r=json.loads(_l); _pid=str(_r["id"]).split("/")[0]
            if _pid not in PROMPTS:
                try: PROMPTS[_pid]=next(t["content"] for t in _r["conversations"]
                                        if t["role"]=="user")
                except StopIteration: pass
N_BSP=int(globals().get("N_BSP",48)); MAX_NEW=int(globals().get("MAX_NEW",96))
CHUNK=int(globals().get("CHUNK",16)); TEMP=float(globals().get("TEMP",1.0))
SEED=int(globals().get("SEED",20260814))
MIN_BSP=int(globals().get("MIN_BSP",20))
N_ABL=int(globals().get("N_ABL",48))
N_KETTEN=int(globals().get("N_KETTEN",3))
N_PRUEF=int(globals().get("N_PRUEF",8))
WIEDERHOLUNG=int(globals().get("WIEDERHOLUNG",0))
def saat(zweck,schl):
    h=2166136261
    for c in (zweck+"/"+schl).encode():
        h=((h^c)*16777619)&0xFFFFFFFF
    return (SEED+1000003*WIEDERHOLUNG+h)%(2**31-1)
def prompt_text(u):
    return "<|im_start|>user\n"+u+"<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
ZIEL_ID=globals().get("ZIEL_ID","") or next(p for p in PROMPTS if PHRASE in PROMPTS[p])
ROH_PROMPT=PROMPTS[ZIEL_ID]
if tokenizer.pad_token_id is None: tokenizer.pad_token=tokenizer.eos_token
tokenizer.padding_side="left"
# ---------------- 0  Architektur --------------------------------------------
print("="*80); print("0  ARCHITEKTUR"); print("="*80)
cfg=model.config
RX=re.compile(r"^(?:model\.)?(?:language_model\.)?(?:model\.)?layers\.(\d+)\.mlp\.experts$")
EXPM={}
for nm,mod in model.named_modules():
    m=RX.match(nm)
    if m: EXPM[int(m.group(1))]=mod
ARCH_OK=bool(EXPM)
if ARCH_OK:
    e0=EXPM[min(EXPM)]
    GU=e0.gate_up_proj; INTER=int(e0.intermediate_dim); NEXP=int(GU.shape[0])
    TOPK=int(cfg.num_experts_per_tok)
    ARCH_OK=(GU.ndim==3 and GU.shape[1]==2*INTER and GU.shape[2]==cfg.hidden_size)
    print("  %d Schichten | %d Experten je Schicht | top-%d | %d Paare gesamt"
          %(len(EXPM),NEXP,TOPK,len(EXPM)*NEXP))
    print("  Formen wie erwartet: %s"%("ja" if ARCH_OK else "NEIN"))
if not ARCH_OK:
    KURVE_RESULTS=dict(verdict="ARCHITEKTUR-NICHT-GEFUNDEN",arch_ok=False)
    wc_save_all(); print(""); print("VERDIKT: ARCHITEKTUR-NICHT-GEFUNDEN"); raise SystemExit(0)
# ---------------- Werkzeuge ---------------------------------------------------
def zieh(text,n,startwert):
    aus=[]
    for b0 in range(0,n,CHUNK):
        b=min(CHUNK,n-b0)
        enc=tokenizer([text]*b,return_tensors="pt",padding=True).to(model.device)
        torch.manual_seed(startwert+b0)
        with torch.no_grad():
            g=model.generate(**enc,do_sample=True,temperature=TEMP,top_p=1.0,top_k=0,
                             repetition_penalty=1.0,max_new_tokens=MAX_NEW,
                             pad_token_id=tokenizer.pad_token_id)
        for j in range(b):
            aus.append(tokenizer.decode(g[j,enc["input_ids"].shape[1]:],skip_special_tokens=True))
    return aus
class Maske:
    """Setzt den Router-Anteil ganzer Experten auf null. Reiner Vorwaerts-Haken:
       kein Sicherungsabzug, wirkt an JEDER Position, restlos umkehrbar."""
    def __init__(self,verboten):
        self.verboten={l:set(v) for l,v in verboten.items() if v}
        self.griffe=[]; self.pruefen=False
        self.getroffen=0; self.rest=0.0; self.formen=None
    def _mach(self,bad):
        def h(mod,args):
            idx=args[1]; w=args[2]
            if idx.shape!=w.shape:
                self.formen=(tuple(idx.shape),tuple(w.shape)); return None
            tr=torch.isin(idx,bad)
            neu=w.masked_fill(tr,0.0)
            if self.pruefen:
                self.getroffen+=int(tr.sum().item())
                if bool(tr.any()):
                    self.rest=max(self.rest,float(neu[tr].abs().max().item()))
            return (args[0],idx,neu)+tuple(args[3:])
        return h
    def __enter__(self):
        for l,vs in self.verboten.items():
            W=EXPM[l].gate_up_proj
            bad=torch.tensor(sorted(vs),device=W.device,dtype=torch.long)
            self.griffe.append(EXPM[l].register_forward_pre_hook(self._mach(bad)))
        return self
    def __exit__(self,*a):
        for g in self.griffe: g.remove()
        self.griffe=[]
        return False
def nur_logits(text):
    ids=tokenizer(text,return_tensors="pt").input_ids.to(model.device)
    with torch.no_grad():
        return model(ids).logits[0,-1].float().cpu().numpy()
def mit_maske(verboten,text,n,startwert,pruef_texte,pruef_logits):
    """Die gesperrten Plaetze werden ueber DIESELBEN Texte gezaehlt, auf denen
       die Dosis angeglichen wurde - Prompt UND Antwort.

       Zuerst stand hier ein einzelner Prompt, waehrend die Angleichung ueber
       Prompt+Antwort lief. Geplant und gemessen wichen dadurch um mehr als
       das Doppelte voneinander ab (33 gegen 17 Plaetze), ohne dass eine der
       beiden Zahlen falsch ausgesehen haette."""
    with Maske(verboten) as M:
        M.pruefen=True
        lg=[nur_logits(t) for t in pruef_texte]
        M.pruefen=False
        if M.formen is not None:
            raise RuntimeError("Router-Formen passen nicht: idx %s, gewichte %s"%M.formen)
        wirk=max(float(np.abs(a-b).max()) for a,b in zip(lg,pruef_logits))
        aus=zieh(text,n,startwert)
    return aus,M.getroffen,M.rest,wirk
def plaetze_ganz(texte):
    z=collections.Counter()
    for text in texte:
        ids=tokenizer(text,return_tensors="pt").input_ids.to(model.device)
        fang={}
        def mach(l):
            def h(mod,args): fang[l]=args[1].detach(); return None
            return h
        hs=[EXPM[l].register_forward_pre_hook(mach(l)) for l in EXPM]
        try:
            with torch.no_grad(): model(ids)
        finally:
            for h in hs: h.remove()
        for l,idx in fang.items():
            for e in idx.reshape(-1).tolist(): z[(l,int(e))]+=1
    return z
def hole_routing(text):
    ids=tokenizer(text,return_tensors="pt").input_ids.to(model.device)
    fang={}
    def mach(l):
        def h(mod,args): fang[l]=args[1].detach(); return None
        return h
    hs=[EXPM[l].register_forward_pre_hook(mach(l)) for l in EXPM]
    try:
        with torch.no_grad():
            o=model(ids[:,:-1],use_cache=True); fang.clear()
            model(ids[:,-1:],past_key_values=o.past_key_values,use_cache=True)
    finally:
        for h in hs: h.remove()
    r=set()
    for l,idx in fang.items():
        for e in idx.reshape(-1,idx.shape[-1])[-1].tolist(): r.add((l,int(e)))
    return sorted(r)
# ---------------- Messkette ------------------------------------------------
N_SCHRITT=int(globals().get("N_SCHRITT",96))
VERWERFEN=int(globals().get("VERWERFEN",6))
N_WIEDER=int(globals().get("N_WIEDER",10))
N_PAAR=int(globals().get("N_PAAR",12))
N_AA=int(globals().get("N_AA",6))
STAPEL=int(globals().get("STAPEL",16))
STUFEN=tuple(globals().get("STUFEN",(50,200,800)))
DELTA_ANTEIL=float(globals().get("DELTA_ANTEIL",0.01))
PERM=int(globals().get("PERM",2000))
class Zwang:
    """Erzwingt das Routing: ersetzt die Indextafel des Routers.

       Anders als Maske aus Phase 12, die nur die GEWICHTE auf null setzt und
       den Befehlsstrom unangetastet laesst. Zwang veraendert wirklich, welche
       Experten laufen - und macht die Antwort damit unbrauchbar. Das ist
       Absicht: hier wird die Uhr gelesen, nie der Text."""
    def __init__(self,tafel):
        self.tafel=tafel; self.griffe=[]; self.gesetzt=0
    def __enter__(self):
        for l,idx in self.tafel.items():
            def mach(t):
                def h(mod,args):
                    neu=t.to(args[1].device)[:args[1].shape[0]] if t.shape[0]>=args[1].shape[0] else t.to(args[1].device)
                    if neu.shape!=args[1].shape: return None
                    self.gesetzt+=1
                    return (args[0],neu,args[2])+tuple(args[3:])
                return h
            self.griffe.append(EXPM[l].register_forward_pre_hook(mach(idx)))
        return self
    def __exit__(self,*a):
        for g in self.griffe: g.remove()
        self.griffe=[]
        return False
def leere_haken():
    """Haken, die genau so viel arbeiten wie die Routing-Haken und nichts
       veraendern. Der Unterschied zwischen 'ohne Haken' und 'leere Haken'
       ist der Fussabdruck der Sonde selbst."""
    gs=[]
    for l in EXPM:
        def h(mod,args):
            _=args[1].detach()
            return None
        gs.append(EXPM[l].register_forward_pre_hook(h))
    return gs
def schritt_zeiten(ids,mit_haken=True,zwang=None,schlaf=None,fangen=True):
    """Lehrergefuehrtes Durchschreiten: an jeder Stelle wird das BEKANNTE
       Token gefuettert, nie das gezogene.

       Damit sind Inhalt, Laenge, Zahl der Schritte und das Wachsen des
       Schluessel-Wert-Speichers zwischen allen Bedingungen bitgleich - auch
       dann, wenn erzwungenes Routing die Ausgabe zu Unsinn macht.

       Gemessen wird zweimal: Wanduhr um den Schritt und Geraetezeit ueber
       cuda-Ereignisse. Die Ereigniszeiten werden ERST NACH dem Lauf
       ausgelesen; ein elapsed_time() mittendrin waere eine Synchronisation
       und damit Teil der Messung. Kein print in der Schleife - die Ausgabe
       laeuft ueber eine Drive-Einbindung und wuerde zehn Millisekunden
       Netzlatenz mitten in die Reihe schreiben."""
    T=int(ids.shape[1])
    n=min(N_SCHRITT,T-1)
    fang={}; griffe=[]
    if mit_haken and fangen:
        def mach(l):
            def h(mod,args): fang[l]=args[1].detach(); return None
            return h
        griffe=[EXPM[l].register_forward_pre_hook(mach(l)) for l in EXPM]
    elif mit_haken:
        griffe=leere_haken()
    ev=[(torch.cuda.Event(enable_timing=True),torch.cuda.Event(enable_timing=True))
        for _ in range(n)]
    host=[0.0]*n; slots=[None]*n; s_zahl=[0]*n
    kontext=(zwang if zwang is not None else _Nichts())
    try:
        with kontext:
            with torch.no_grad():
                o=model(ids[:,:1],use_cache=True); vergangen=o.past_key_values
                torch.cuda.synchronize()
                for i in range(n):
                    fang.clear()
                    t0=time.perf_counter(); ev[i][0].record()
                    o=model(ids[:,i+1:i+2],past_key_values=vergangen,use_cache=True)
                    if schlaf is not None and schlaf[i]>0:
                        torch.cuda._sleep(int(schlaf[i]*ZYKLEN_JE_US))
                    ev[i][1].record()
                    vergangen=o.past_key_values
                    host[i]=1000.0*(time.perf_counter()-t0)
                    if fangen and fang:
                        w=[]; z=0
                        for l in sorted(fang):
                            r=fang[l].reshape(-1).tolist()
                            w.extend((l,int(e)) for e in set(r)); z+=len(set(r))
                        slots[i]=sorted(set(w)); s_zahl[i]=z
    finally:
        for g in griffe: g.remove()
    torch.cuda.synchronize()
    dev=[float(a.elapsed_time(b)) for a,b in ev]
    k=min(VERWERFEN,max(0,n-8))
    return dict(host=host[k:],dev=dev[k:],slots=slots[k:],s=s_zahl[k:],
                warm_host=host[:k],n=n-k,verworfen=k)
class _Nichts:
    def __enter__(self): return self
    def __exit__(self,*a): return False
def eiche_schlaf(proben=6):
    """torch.cuda._sleep zaehlt ZYKLEN, nicht Mikrosekunden. Wie viele Zyklen
       eine Mikrosekunde sind, haengt an der Taktrate und wird gemessen."""
    z=1000000
    ts=[]
    for _ in range(proben):
        a=torch.cuda.Event(enable_timing=True); b=torch.cuda.Event(enable_timing=True)
        torch.cuda.synchronize(); a.record(); torch.cuda._sleep(z); b.record()
        torch.cuda.synchronize(); ts.append(float(a.elapsed_time(b)))
    ms=float(np.median(ts))
    return z/(1000.0*ms) if ms>0 else 1000.0
def wettbewerb_an(groesse=2048):
    """Hintergrundlast, damit sich zeigt, ob Routing unter Konkurrenz kippt."""
    A=torch.randn(groesse,groesse,device=model.device,dtype=torch.bfloat16)
    B=torch.randn(groesse,groesse,device=model.device,dtype=torch.bfloat16)
    for _ in range(60): A=A@B
    return A
def routing_einmal(ids):
    """Routing an der letzten Position, ein Vorwaertslauf."""
    fang={}
    def mach(l):
        def h(mod,args): fang[l]=args[1].detach(); return None
        return h
    hs=[EXPM[l].register_forward_pre_hook(mach(l)) for l in EXPM]
    try:
        with torch.no_grad(): model(ids)
    finally:
        for h in hs: h.remove()
    r=set()
    for l,idx in fang.items():
        for e in idx.reshape(-1,idx.shape[-1])[-1].tolist(): r.add((l,int(e)))
    return sorted(r)
def tafel_vielfalt(d,zeilen,rnd):
    """Indextafel mit GENAU d verschiedenen Experten ueber alle Zeilen, je
       Zeile acht verschiedene. Das ist die Dosis fuer H1."""
    aus={}
    for l in EXPM:
        pool=rnd.sample(range(NEXP),min(d,NEXP))
        zs=[]
        for _ in range(zeilen):
            zs.append(sorted(rnd.sample(pool,TOPK)) if len(pool)>=TOPK
                      else sorted((pool*TOPK)[:TOPK]))
        aus[l]=torch.tensor(zs,device=model.device,dtype=torch.long)
    return aus
def tafel_menge(menge,zeilen,fueller,rnd):
    """Indextafel, die eine vorgegebene Menge erzwingt - je Schicht auf acht
       aufgefuellt. Fuer H2: gleiche Vielfalt, andere Namen."""
    js=collections.defaultdict(list)
    for l,e in menge: js[l].append(e)
    aus={}
    for l in EXPM:
        h=sorted(set(js.get(l,[])))
        rest=[e for e in fueller if e not in h]
        v=(h+rest)[:TOPK]
        while len(v)<TOPK: v.append((v[-1]+1)%NEXP if v else 0)
        aus[l]=torch.tensor([sorted(set(v))[:TOPK] if len(set(v))>=TOPK else sorted(v)
                             for _ in range(zeilen)],device=model.device,dtype=torch.long)
    return aus
# ---------------- Arme, Menge und der feste Strom ---------------------------
ARME=[("NEU","each service's name","Bezugsarm"),
      ("JA","each service's Japanese name","Kana"),
      ("BR1","each service's Braille name","konstruiert, fremde Schrift"),
      ("MORSE","each service's name written in Morse code","konstruiert, ASCII"),
      ("SR","each service's Serbian name","kyrillisch"),
      ("RU","each service's Russian name","kyrillisch")]
BEZUG="NEU"
ARM_LISTE=[a for a,_,_ in ARME]
MASS={"NEU":"englisch","JA":"kana","SR":"kyrillisch","RU":"kyrillisch",
      "BR1":"braille","MORSE":"morse"}
ARMTEXT={}
for schl,phrase,_ in ARME:
    _t,_ok=setze_arm(ROH_PROMPT,phrase)
    assert _ok,"Phrase %r nicht im Prompt gefunden"%PHRASE
    ARMTEXT[schl]=prompt_text(_t)
MENGE=sorted(set(hole_routing(ARMTEXT["JA"]))-set(hole_routing(ARMTEXT[BEZUG])))
def _gierig(text,n):
    """EIN gieriger Durchlauf. Danach wird nur noch lehrergefuehrt
       durchgeschritten - der Strom steht damit fest, und keine Bedingung
       kann ihn mehr veraendern."""
    ids=tokenizer(text,return_tensors="pt").input_ids.to(model.device)
    with torch.no_grad():
        o=model(ids,use_cache=True); v=o.past_key_values
        letzt=int(o.logits[0,-1].argmax()); aus=[letzt]
        for _ in range(n-1):
            o=model(torch.tensor([[letzt]],device=model.device),
                    past_key_values=v,use_cache=True)
            v=o.past_key_values; letzt=int(o.logits[0,-1].argmax())
            aus.append(letzt)
    return torch.cat([ids,torch.tensor([aus],device=model.device)],dim=1)
ARM_IDS={s:_gierig(ARMTEXT[s],N_SCHRITT+1) for s,_,_ in ARME}
FESTE_IDS=ARM_IDS["JA"]
# ---------------- 0  Pfad und Groessen --------------------------------------
print(""); print("="*80); print("0b  WELCHER PFAD LAEUFT, UND WAS KOSTET ER")
print("="*80)
_e0=EXPM[min(EXPM)]
BYTE_JE_EXP=int(_e0.gate_up_proj[0].numel()*_e0.gate_up_proj.element_size()
                +_e0.down_proj[0].numel()*_e0.down_proj.element_size())
BYTE_JE_SCHRITT=BYTE_JE_EXP*TOPK*len(EXPM)
print("  Umsetzung: %s | forward: %s"
      %(type(_e0).__name__,getattr(getattr(type(_e0),"forward",None),"__qualname__","?")))
print("  ein Experte: %.2f MiB (gate_up %.2f + down %.2f)"
      %(BYTE_JE_EXP/2**20,_e0.gate_up_proj[0].numel()*_e0.gate_up_proj.element_size()/2**20,
        _e0.down_proj[0].numel()*_e0.down_proj.element_size()/2**20))
print("  ein Dekodierschritt bei B=1 adressiert %.2f GB Expertengewicht -"
      %(BYTE_JE_SCHRITT/1e9))
print("  und zwar UNABHAENGIG davon, welche %d es sind."%TOPK)
print("")
print("  Daraus folgt die erste Absage: zwischen zwei Beruehrungen desselben")
print("  Paares stroemen mehr als %.1f GB durch den L2. Wiederverwendung ueber"
      %(BYTE_JE_SCHRITT/1e9*1.75))
print("  benachbarte Tokens ist rechnerisch tot - das war die naheliegende")
print("  Rettung der Ausgangsidee, und sie faellt hier, nicht spaeter.")
_aus=[]
try:
    import warnings as _w
    torch.cuda.set_sync_debug_mode("warn")
    with _w.catch_warnings(record=True) as _c:
        _w.simplefilter("always")
        with torch.no_grad(): model(FESTE_IDS[:,:2])
        _aus=[str(x.message)[:40] for x in _c]
except Exception as _x:
    _aus=None
finally:
    try: torch.cuda.set_sync_debug_mode("default")
    except Exception: pass
print("  Geraet-nach-Host-Synchronisationen je Vorwaertslauf: %s"
      %("nicht messbar" if _aus is None else str(len(_aus))))
# ---------------- 1  Fester Tokenstrom --------------------------------------
print(""); print("="*80); print("1  EIN FESTER TOKENSTROM FUER ALLE ZEITMESSUNGEN")
print("="*80)
print("  Erzeugt wird EINMAL, danach wird nur noch lehrergefuehrt")
print("  durchgeschritten: an jeder Stelle das bekannte Token, nie das")
print("  gezogene. Inhalt, Laenge und das Wachsen des Schluessel-Wert-")
print("  Speichers sind damit zwischen allen Bedingungen bitgleich - auch")
print("  dann, wenn erzwungenes Routing die Ausgabe zu Unsinn macht.")
print("  Schritte: %d, davon %d verworfen (Aufwaermen)"%(N_SCHRITT,VERWERFEN))
# ---------------- 2  Eichung der Uhr ----------------------------------------
print(""); print("="*80); print("2  EICHUNG: WAS KANN DIE UHR SEHEN? (EPSILON)")
print("="*80)
ZYKLEN_JE_US=eiche_schlaf()
print("  torch.cuda._sleep: %.0f Zyklen je Mikrosekunde"%ZYKLEN_JE_US)
rnd=random.Random(saat("leiter","alle"))
_z=[]; _m=[]
for s in (0,)+STUFEN:
    for _ in range(4):
        sch=[s if rnd.random()<0.5 else 0 for _ in range(N_SCHRITT)]
        d=schritt_zeiten(FESTE_IDS,mit_haken=False,schlaf=sch,fangen=False)
        for t,x in zip(d["dev"],sch[VERWERFEN:]): _z.append(t); _m.append(x)
EPS,LEITER=leiter(_z,_m,STUFEN,perm=400,rnd=random.Random(saat("leiterp","alle")))
print("  %-10s %12s %10s"%("Stufe","gemessen ms","p"))
for s,dd,pp in LEITER:
    print("  %-10d %12s %10s"%(s,"-" if dd is None else "%.4f"%dd,
                               "-" if pp is None else "%.4f"%pp))
print("  EPSILON = %s"%("NICHT ERREICHT" if EPS is None else "%d us"%EPS))
if EPS is None:
    print("")
    print("  Die Kette sieht nicht einmal %d us. Damit ist keine Aussage ueber"%max(STUFEN))
    print("  Mikrosekunden zulaessig, und der ganze Zeitarm entfaellt.")
# ---------------- 3  Wiederholbarkeit ---------------------------------------
print(""); print("="*80); print("3  WIEDERHOLT SICH DAS ZEITPROFIL? (%d Laeufe)"%N_WIEDER)
print("="*80)
WIEDER=[schritt_zeiten(FESTE_IDS,mit_haken=False,fangen=False) for _ in range(N_WIEDER)]
ICC,ICC_P=icc_haelften([w["dev"] for w in WIEDER],perm=200,
                       rnd=random.Random(saat("icc","alle")))
print("  Split-half nach TRENDABZUG: r = %s | p = %s"
      %("-" if ICC is None else "%.3f"%ICC,"-" if ICC_P is None else "%.4f"%ICC_P))
print("  (ohne Abzug maesse die Zahl nur den wachsenden Speicher - der ist in")
print("   allen Wiederholungen derselbe und ergaebe auch bei reinem Rauschen")
print("   fast eins)")
_med=float(np.median([x for w in WIEDER for x in w["dev"]]))
DELTA=DELTA_ANTEIL*_med
print("  Mittlere Schrittzeit %.3f ms | Schranke delta = %.4f ms (%.0f %%)"
      %(_med,DELTA,100*DELTA_ANTEIL))
_gap=[h-d for w in WIEDER for h,d in zip(w["host"],w["dev"])]
print("  Wanduhr minus Geraetezeit: Median %.3f ms - das ist der Host-Anteil"
      %float(np.median(_gap)))
# ---------------- 4  Stoert die Sonde? --------------------------------------
print(""); print("="*80); print("4  IST DIE SONDE DAS SIGNAL?")
print("="*80)
print("  Drei Bedingungen auf demselben Strom, verschraenkt gezogen:")
print("    ohne     keine Haken")
print("    leer     Haken, die genau so viel arbeiten und nichts aendern")
print("    maske    die 42 gesperrt. Maske setzt nur GEWICHTE auf null und")
print("             laesst den Befehlsstrom unangetastet - dieselben Gruppen,")
print("             dieselben Kernel, dieselben %.2f GB. Sie veraendert das"%(BYTE_JE_SCHRITT/1e9))
print("             Verhalten vollstaendig (Phase 15: 100 auf 9 %%) und die")
print("             Arbeit gar nicht. Genau deshalb ist sie hier eine")
print("             ATTRAPPE und keine Behandlung.")
SONDE={}
for nm in ("ohne","leer","maske"):
    xs=[]
    for _ in range(6):
        if nm=="maske":
            with Maske(nach_schicht(MENGE)):
                d=schritt_zeiten(FESTE_IDS,mit_haken=False,fangen=False)
        else:
            d=schritt_zeiten(FESTE_IDS,mit_haken=(nm=="leer"),fangen=False)
        xs.extend(d["dev"])
    SONDE[nm]=xs
    print("  %-6s Median %.4f ms"%(nm,float(np.median(xs))))
S_HAKEN=tost(SONDE["leer"],SONDE["ohne"],DELTA)
S_MASKE=tost(SONDE["maske"],SONDE["leer"],DELTA)
print("  Haken gegen ohne : %-14s (d=%s ms)"
      %(S_HAKEN[0],"-" if S_HAKEN[1] is None else "%.4f"%S_HAKEN[1]))
print("  Maske gegen leer : %-14s (d=%s ms)"
      %(S_MASKE[0],"-" if S_MASKE[1] is None else "%.4f"%S_MASKE[1]))
SONDE_URTEIL="VERSCHIEDEN" if "VERSCHIEDEN" in (S_HAKEN[0],S_MASKE[0]) else S_HAKEN[0]
# ---------------- 5  Kippt Routing unter Last? ------------------------------
print(""); print("="*80); print("5  BERUEHRT LAST DAS ROUTING?")
print("="*80)
print("  Die Ausgangsidee liest 'Experte e bevorzugt die Hochlastphase'. Das")
print("  braucht Last -> Routing. Routing ist aber eine Funktion von Eingabe")
print("  und Gewichten; Taktrate und Temperatur gehen ins top-k nicht ein.")
print("  Der einzige Weg waere numerische Nichtdeterminiertheit nahe an")
print("  Gleichstaenden. Also: dasselbe Routing zweimal, einmal unter")
print("  erzwungener Konkurrenz.")
KIPP=[]
for _ in range(4):
    a=routing_einmal(FESTE_IDS)
    _w=wettbewerb_an(); b=routing_einmal(FESTE_IDS); del _w
    KIPP.append(kippzahl(a,b))
_n_paare=len(EXPM)*NEXP
print("  gekippte Plaetze je Durchgang: %s (von %d)"%(KIPP,_n_paare))
print("  Schranke: %.2e - beobachtete null ist eine SCHRANKE, kein Beweis."
      %(1.0/(_n_paare*max(1,len(KIPP)))))
LAST_ROUTING="LASTFEST" if sum(KIPP)==0 else "LASTLABIL"
# ---------------- 6  H1  Sieht die Zeit die VIELFALT? -----------------------
print(""); print("="*80); print("6  H1: ERZWUNGENE VIELFALT")
print("="*80)
print("  Bei B=1 ist die Vielfalt konstant %d - jede Schicht trifft acht"%(TOPK*len(EXPM)))
print("  Experten, egal welche. Erst im Stapel wird sie zur Stellgroesse: %d"%STAPEL)
print("  gleiche Zeilen, aber eine erzwungene Tafel mit D verschiedenen")
print("  Experten je Schicht. Gleiche Zeilen sind Absicht - so ist die")
print("  natuerliche Vielfalt genau acht und der Stapelzerfall, der sonst")
print("  jede Stapelmessung beherrscht, faellt weg.")
print("")
print("  Zwei Ausgaenge, beide aufschlussreich: zahlt die Umsetzung je")
print("  NICHTLEERER GRUPPE, ist kappa positiv; holt sie fuer jeden Platz")
print("  eine Zeile, ist kappa null - und das ist die Diagnose 'die Laufzeit")
print("  zahlt fuer Plaetze, nicht fuer verschiedene Experten'.")
STAPEL_IDS=FESTE_IDS.repeat(STAPEL,1)
D_STUFEN=[d for d in (TOPK,2*TOPK,4*TOPK,8*TOPK,TOPK*STAPEL) if d<=NEXP]
rnd=random.Random(saat("vielfalt","alle"))
H1_X=[]; H1_Y=[]; H1_OK=True
_pl=[(d,r) for d in D_STUFEN for r in range(3)]
rnd.shuffle(_pl)
# Gemessen wird gegen die ERREICHTE Vielfalt, nicht gegen die bestellte.
# Mehr als TOPK*B verschiedene passen nicht in einen Stapel; wer trotzdem auf
# die Bestellzahl regressiert, mittelt zwei gleiche Zustaende unter
# verschiedenen Namen und drueckt die Steigung gegen null. In der Probewelt
# mit vier Zeilen sattigte D bei 16, und kappa fiel von 0.02 auf -0.0004.
for d,_ in _pl:
    try:
        with Zwang(tafel_vielfalt(d,STAPEL,rnd)) as Z:
            dd=schritt_zeiten(STAPEL_IDS,mit_haken=True,fangen=True)
        if Z.gesetzt==0: H1_OK=False
        _s=[x for x in dd["s"] if x>0]
        H1_X.append(float(np.mean(_s)) if _s else float(d*len(EXPM)))
        H1_Y.append(float(np.median(dd["dev"])))
    except Exception as _x:
        H1_OK=False; print("  Zwang gescheitert: %s"%str(_x)[:60]); break
if H1_OK and len(H1_X)>=6:
    KAPPA,K_LO,K_HI=steigung(H1_X,H1_Y,proben=400,rnd=random.Random(saat("kappa","alle")))
    _,K_P=blocknull(H1_Y,H1_X,perm=PERM,rnd=random.Random(saat("kappan","alle")))
    print("  %-14s %14s"%("S erreicht","Median ms"))
    for d in sorted(set(H1_X)):
        print("  %-14.1f %14.4f"%(d,float(np.median([y for x,y in zip(H1_X,H1_Y) if x==d]))))
    print("  kappa = %s ms je Einheit S  [%s, %s] | p = %s"
          %("-" if KAPPA is None else "%.5f"%KAPPA,
            "-" if K_LO is None else "%.5f"%K_LO,
            "-" if K_HI is None else "%.5f"%K_HI,
            "-" if K_P is None else "%.4f"%K_P))
    if KAPPA is None:
        print("  Die erreichte Vielfalt schwankt gar nicht - der Stapel ist zu")
        print("  klein oder der Zwang greift nicht. Ohne Streuung in S gibt es")
        print("  keine Steigung, und H1 ist nicht beantwortet.")
else:
    KAPPA=K_LO=K_HI=K_P=None
    print("  H1 entfaellt: Zwang greift nicht (ZWANG-WIRKUNGSLOS)")
# ---------------- 7  H2  Sieht die Zeit die IDENTITAET? ---------------------
print(""); print("="*80); print("7  H2: GLEICHE VIELFALT, ANDERE NAMEN")
print("="*80)
print("  Beide Bedingungen erzwingen acht verschiedene je Schicht - gleiche")
print("  Formen, gleiche Hakenkosten, gleiche %.2f GB. Verschieden sind nur"%(BYTE_JE_SCHRITT/1e9))
print("  die NAMEN: A enthaelt die 42 in ihren Schichten, B je Wiederholung")
print("  einen FRISCHEN ratengleichen Partnersatz. Frisch je Wiederholung -")
print("  das beantwortet zugleich den Vorbehalt aus Phase 15, dass dort nur")
print("  EIN zufaelliges Achtel gezogen wurde.")
print("")
print("  Gezaehlt wird auf LAUFPAAREN, nicht auf Schritten. Die")
print("  Vorzeichenumkehr je Schritt unterstellt unabhaengige Vorzeichen; in")
print("  der Probewelt mit Nachbarkorrelation 0.7 ergab das 33 %% Fehlalarm")
print("  bei nominal 5. Die A/A-Paare eichen das im Lauf selbst.")
_kand=[e for e in range(NEXP)]
H2_AB=[]; H2_AA=[]
rnd=random.Random(saat("ident","alle"))
def _lauf_tafel(menge):
    with Zwang(tafel_menge(menge,1,_kand,rnd)):
        return schritt_zeiten(FESTE_IDS,mit_haken=False,fangen=False)["dev"]
if H1_OK:
    for i in range(N_PAAR):
        partner=[(l,rnd.choice([e for e in range(NEXP)
                                if (l,e) not in set(MENGE)])) for l,_ in MENGE]
        a=_lauf_tafel(MENGE); b=_lauf_tafel(partner)
        if i%2: a,b=b,a
        H2_AB.append(float(np.median(np.asarray(a)-np.asarray(b))))
    for _ in range(N_AA):
        a=_lauf_tafel(MENGE); b=_lauf_tafel(MENGE)
        H2_AA.append(float(np.median(np.asarray(a)-np.asarray(b))))
    D_AB,P_AB=gepaart(H2_AB,perm=PERM,rnd=random.Random(saat("h2","alle")))
    D_AA,P_AA=gepaart(H2_AA,perm=PERM,rnd=random.Random(saat("h2aa","alle")))
    IDENT=tost(H2_AB,[0.0]*len(H2_AB),DELTA) if len(H2_AB)>=3 else ("UNENTSCHIEDEN",None,None)
    print("  A/B: Median der Paardifferenzen %s ms | p = %s"
          %("-" if D_AB is None else "%.5f"%D_AB,"-" if P_AB is None else "%.4f"%P_AB))
    print("  A/A: Median %s ms | p = %s   <- die Eichung"
          %("-" if D_AA is None else "%.5f"%D_AA,"-" if P_AA is None else "%.4f"%P_AA))
    if P_AA is not None and P_AA<0.05:
        print("  ACHTUNG: schon A gegen A lehnt ab. Die Null ist nicht geeicht,")
        print("  und der A/B-Befund ist wertlos.")
        IDENT=("NULL-NICHT-GEEICHT",D_AB,P_AB)
    if D_AB is not None:
        _sk=("%.1f"%(abs(D_AB)*1000.0/EPS) if EPS else "-")
        print("  Schranke: |d| = %.1f us = %.2f %% der Schrittzeit = %s x EPSILON"
              %(abs(D_AB)*1000.0,100.0*abs(D_AB)/_med,_sk))
    print("  Urteil: %s"%IDENT[0])
else:
    D_AB=P_AB=D_AA=P_AA=None; IDENT=("ENTFAELLT",None,None)
    print("  entfaellt - ohne wirksamen Zwang gibt es keine feste Vielfalt")
# ---------------- 8  H3  Der Lastzaehler je Arm -----------------------------
print(""); print("="*80); print("8  H3: DER LASTZAEHLER S(t) JE ARM")
print("="*80)
print("  S(t) ist rauschfrei - er kommt aus dem Routing, nicht aus der Uhr.")
print("  Die Laeufe wechseln sich REIHUM ab statt in Armbloecken, damit die")
print("  Drift nicht ganz auf einen Arm faellt. Die Null vertauscht das")
print("  ARMETIKETT ueber die LAEUFE: 20 Laeufe sind die unabhaengigen")
print("  Einheiten, 20 mal 96 Tokens waeren geborgte Freiheitsgrade.")
H3={}; H3_LAUF=[]; H3_ARM=[]
for r in range(4):
    for schl in ARM_LISTE:
        d=schritt_zeiten(ARM_IDS[schl],mit_haken=True,fangen=True)
        s=[x for x in d["s"] if x>0]
        if not s: continue
        H3.setdefault(schl,[]).append(d)
        H3_LAUF.append(float(np.mean(s))); H3_ARM.append(schl)
print("  %-6s %10s %12s %12s"%("Arm","Laeufe","S im Mittel","ms im Mittel"))
for schl in ARM_LISTE:
    if schl not in H3: continue
    ss=[x for d in H3[schl] for x in d["s"] if x>0]
    tt=[x for d in H3[schl] for x in d["dev"]]
    print("  %-6s %10d %12.1f %12.4f"%(schl,len(H3[schl]),np.mean(ss),np.mean(tt)))
A_SPANN,A_P,A_BODEN=armnull(H3_LAUF,H3_ARM,perm=PERM,
                            rnd=random.Random(saat("arm","alle")))
print("  Spannweite der Armmittel: %s | p = %s (Untergrenze %s)"
      %("-" if A_SPANN is None else "%.2f"%A_SPANN,
        "-" if A_P is None else "%.4f"%A_P,"-" if A_BODEN is None else "%.4f"%A_BODEN))
print("  Bei B=1 ist S konstant %d - eine Spannweite ueber null waere hier"%(TOPK*len(EXPM)))
print("  ein Messfehler, keine Entdeckung.")
# ---------------- 9  H4  Gibt es ueberhaupt einen Traeger? ------------------
print(""); print("="*80); print("9  H4: GIBT ES EINEN TRAEGER MIT PERIODE > 1 TOKEN?")
print("="*80)
print("  Erst hier entscheidet sich, ob die Ausgangsidee ueberhaupt eine")
print("  Koordinate hat. Eine Phase braucht etwas Periodisches; beim")
print("  Dekodieren wiederholt sich genau ein Ereignis - das Token. Ob es")
print("  darueber hinaus einen Traeger gibt, wird gemessen und nicht")
print("  angenommen. Die Null vertauscht die Residuen INNERHALB des Laufs:")
print("  das zerstoert jede zeitliche Ordnung und behaelt die Verteilung.")
TRAEGER=[]
for i,w in enumerate(WIEDER):
    p,wert,sw=traeger(w["dev"],list(range(len(w["dev"]))),perm=200,
                      rnd=random.Random(saat("traeger","%d"%i)))
    TRAEGER.append((p,wert,sw))
_gef=[p for p,_,_ in TRAEGER if p is not None]
print("  %d von %d Laeufen zeigen einen Gipfel ueber der Null"%(len(_gef),len(TRAEGER)))
if _gef:
    from collections import Counter as _C
    _h=_C(_gef).most_common(3)
    print("  haeufigste Perioden: %s"%", ".join("%d (%dx)"%(a,b) for a,b in _h))
    TRAEGER_P=_h[0][0] if _h[0][1]>=max(2,len(TRAEGER)//2) else None
else:
    TRAEGER_P=None
print("  Ergebnis: %s"%("KEIN-TRAEGER" if TRAEGER_P is None else "TRAEGER-BEI-%d"%TRAEGER_P))
if TRAEGER_P is None:
    print("  Damit ist phi(t) nicht definiert. Nicht 'wir haben nichts")
    print("  gefunden', sondern: die Koordinate, auf der die Frage steht,")
    print("  existiert in dieser Anordnung nicht.")
# ---------------- 10  H5  Prefill gegen Decode ------------------------------
print(""); print("="*80); print("10  H5: LIEST DER PREFILL DASSELBE ROUTING WIE DAS DEKODIEREN?")
print("="*80)
print("  Eine Pruefung der frueheren Phasen. 14 bis 18 lesen das Routing aus")
print("  EINEM lehrergefuehrten Vorwaertslauf ueber Prompt und Antwort. Beim")
print("  Erzeugen laeuft das Modell aber Schritt fuer Schritt mit")
print("  Schluessel-Wert-Speicher. Stimmen die beiden nicht ueberein, haengt")
print("  jede Verknuepfung der frueheren Befunde an einem Routing, das nie")
print("  gelaufen ist. Zufallsueberlappung waere %.2f von %d."
      %(TOPK*TOPK/float(NEXP),TOPK))
H5={}
for schl in ARM_LISTE:
    ids=ARM_IDS[schl]
    fang={}
    def _m(l):
        def h(mod,args): fang[l]=args[1].detach(); return None
        return h
    hs=[EXPM[l].register_forward_pre_hook(_m(l)) for l in EXPM]
    try:
        with torch.no_grad(): model(ids)
    finally:
        for h in hs: h.remove()
    vor={}
    for l,idx in fang.items():
        r=idx.reshape(-1,idx.shape[-1])
        for t in range(r.shape[0]): vor[(l,t)]=set(int(x) for x in r[t].tolist())
    d=schritt_zeiten(ids,mit_haken=True,fangen=True)
    gleich=0; ges=0; ueber=[]
    for i,sl in enumerate(d["slots"]):
        if not sl: continue
        t=i+d["verworfen"]+1
        je=collections.defaultdict(set)
        for l,e in sl: je[l].add(e)
        for l,s in je.items():
            v=vor.get((l,t))
            if v is None: continue
            ges+=1; ueber.append(len(s&v))
            if s==v: gleich+=1
    H5[schl]=dict(anteil=(gleich/float(ges) if ges else None),
                  ueberlappung=(float(np.mean(ueber)) if ueber else None),n=ges)
    print("  %-6s gleiche Achtermengen %s | mittlere Ueberlappung %s von %d"
          %(schl,"-" if H5[schl]["anteil"] is None else "%.1f %%"%(100*H5[schl]["anteil"]),
            "-" if H5[schl]["ueberlappung"] is None else "%.2f"%H5[schl]["ueberlappung"],TOPK))
_an=[H5[s]["anteil"] for s in H5 if H5[s]["anteil"] is not None]
PFAD_GLEICH=bool(_an and min(_an)>=0.95)
print("  Ergebnis: %s"%("ROUTING-PFADGLEICH" if PFAD_GLEICH else "ROUTING-PFADVERSCHIEDEN"))
if not PFAD_GLEICH:
    print("  Das ist ein Befund ueber die frueheren Phasen, nicht ueber diese.")
# ---------------- Urteil ----------------------------------------------------
GATES=[("ARCHITEKTUR-NICHT-GEFUNDEN",ARCH_OK),
       ("ZWANG-WIRKUNGSLOS",bool(H1_OK))]
CODE=urteil_takt(EPS,ICC_P,SONDE_URTEIL,K_P,IDENT[0],TRAEGER_P,GATES)
print(""); print("="*80); print("VERDIKT: %s"%CODE); print("="*80)
if CODE=="UHR-BLIND":
    print("  Die Messkette findet nicht einmal die eingespeiste Verzoegerung")
    print("  wieder. Jede Zahl ueber Mikrosekunden waere geraten.")
elif CODE=="KEIN-TAKT":
    print("  Das Zeitprofil wiederholt sich nicht. Eine Reihe ohne")
    print("  Wiederholbarkeit kann keinen Armunterschied tragen, bei keinem p.")
elif CODE=="SONDE-IST-SIGNAL":
    print("  Die Haken bewegen die Uhr staerker als die Schranke. Jeder")
    print("  gefundene Effekt waere der eigene Fussabdruck.")
elif CODE=="ZEIT-SIEHT-VIELFALT-NICHT-IDENTITAET":
    print("  Die Zeit folgt der ZAHL verschiedener Experten (kappa = %s ms"
          %("-" if KAPPA is None else "%.5f"%KAPPA))
    print("  je Einheit) und NICHT ihren Namen. Genau das sagt die Bauart")
    print("  voraus: die Schleife laeuft einmal je getroffenem Experten, und")
    print("  jeder kostet dieselben %.2f MiB."%(BYTE_JE_EXP/2**20))
elif CODE=="ZEIT-SIEHT-IDENTITAET":
    print("  Bei gleicher Vielfalt bewegen verschiedene NAMEN die Uhr. Das")
    print("  waere ueberraschend und verlangt eine Wiederholung mit")
    print("  WIEDERHOLUNG=1, bevor es etwas wert ist.")
else:
    print("  Weder Vielfalt noch Identitaet bewegen die Uhr ueber die")
    print("  Aufloesung hinaus. Die Schranke steht oben.")
print("")
print("  ZUR AUSGANGSFRAGE")
print("  Eine Phasenkoordinate phi(t) je Feuerereignis gibt es hier nicht:")
print("  %s, und innerhalb eines Tokens ist die feinste"
      %("kein Traeger gefunden" if TRAEGER_P is None else "Traeger bei %d"%TRAEGER_P))
print("  trennbare Zeitkoordinate die SCHICHT, nicht der Experte - alle acht")
print("  einer Schicht laufen am selben Punkt des Durchlaufs. Was von der")
print("  Idee bleibt und gemessen wurde, ist die Vielfalt.")
print("  Last beruehrt das Routing: %s (%d gekippte Plaetze in %d Durchgaengen)"
      %(LAST_ROUTING,sum(KIPP),len(KIPP)))
print("")
print("  WAS DIESER LAUF NICHT SAGT")
print("  Nichts ueber Qwen3.6. Die Zeit je Schritt ist eine Eigenschaft der")
print("  LAUFZEIT, nicht des Netzes: dieselbe Auswahl unter einem fusionierten")
print("  Kernel, unter CUDA-Graphen oder unter vLLM ergaebe einen ganz anderen")
print("  Takt. Jeder Zeitbefund gehoert in einen Abschnitt ueber transformers")
print("  %s auf sm80, nicht in einen ueber das Modell."%getattr(__import__("transformers"),"__version__","?"))
print("  Nichts unterhalb von EPSILON. Nichts ueber eine Phase unterhalb der")
print("  Schicht - die gibt es nicht. Und H1/H2 sagen nichts ueber Verhalten:")
print("  erzwungenes Routing zerstoert die Ausgabe, das ist ihr Zweck.")
TAKT_RESULTS=dict(verdict=CODE,arch_ok=bool(ARCH_OK),prompt_id=ZIEL_ID,
    wiederholung=WIEDERHOLUNG,n_schritt=N_SCHRITT,verworfen=VERWERFEN,
    byte_je_experte=BYTE_JE_EXP,byte_je_schritt=BYTE_JE_SCHRITT,
    epsilon_us=EPS,leiter=[[s,d,p] for s,d,p in LEITER],
    icc=ICC,icc_p=ICC_P,delta_ms=DELTA,median_ms=_med,
    sonde={k:float(np.median(v)) for k,v in SONDE.items()},
    sonde_haken=list(S_HAKEN),sonde_maske=list(S_MASKE),sonde_urteil=SONDE_URTEIL,
    kipp=KIPP,last_routing=LAST_ROUTING,
    h1_x=H1_X,h1_y=H1_Y,kappa=KAPPA,kappa_lo=K_LO,kappa_hi=K_HI,kappa_p=K_P,
    h2_ab=H2_AB,h2_aa=H2_AA,h2_d=D_AB,h2_p=P_AB,h2_aa_p=P_AA,identitaet=list(IDENT),
    h3_lauf=H3_LAUF,h3_arm=H3_ARM,arm_spann=A_SPANN,arm_p=A_P,arm_boden=A_BODEN,
    traeger=[[p,w,s] for p,w,s in TRAEGER],traeger_periode=TRAEGER_P,
    h5={s:H5[s] for s in H5},pfad_gleich=PFAD_GLEICH,menge=[list(q) for q in MENGE])
wc_save("takt_stufen",dict(h1=dict(zip(map(str,H1_X),H1_Y)),h2=H2_AB,h2_aa=H2_AA))
wc_save_all()
